# CrossFuse v3.3 — Cross-Attention-Fused, Identity-Disjoint, Bias-Checked Deepfake Detector

This is a from-the-ground-up bug-fix and generalization pass over `deepfake v3.2.ipynb`, driven by a
full audit of that notebook's own outputs (train/val AUC logs, the DFDC confusion matrix, and an
in-the-wild upload that returned a false-positive audio verdict alongside a
`PySoundFile failed. Trying audioread instead` warning).

**What v3.2's own numbers showed:**
- The video head **overfits**: train AUC 0.971 vs val AUC 0.788 (a 0.18 gap, still widening at epoch 15
  of a fixed 15-epoch schedule — the best checkpoint was actually epoch 11).
- The video head **does not transfer**: DFDC zero-shot AUC 0.627, on only 154 clips (95% CI ≈ ±0.09).
- The audio head's 0.9986 val AUC is **shortcut learning, not skill** — val AUC exceeds train AUC, a
  classic sign the head found the training set's TTS generator fingerprint rather than a general notion
  of "fake speech." The in-the-wild upload's 71.8% audio-fake score, decoded through a different librosa
  backend than any training tensor, is the proof.
- **~290 clips were silently missing** from the dataset (1,706 planned → 1,416 actual) due to a
  filename-collision bug in the caching key.
- **Every reported number was optimistically biased** — thresholds, ECE, the classification report and
  the 4-way attribution matrix were all fit/reported on the same val set used for checkpoint selection.

**What v3.3 changes, in one sentence each:**
1. **Cross-attention is now load-bearing.** Bidirectional cross-attention (video↔audio) feeds a third
   `sync_head`, trained self-supervised on shuffled AV pairs, that joins the final verdict — so
   "Cross-Attention Fusion" describes what the model actually does, not just an unused auxiliary term.
2. **Masked stats pooling + padding masks** replace naive mean-pooling over zero-padded clips, so the
   video head sees temporal variance (where deepfake flicker lives) instead of averaging it away.
3. **Collision-proof extraction manifest**, batched 2-GPU feature extraction (5–10x faster), a unified
   `ffmpeg`-based audio front-end used identically everywhere, full-clip time-aligned audio/video, and
   compression/quality-augmented replicas aimed directly at the DFDC domain gap.
4. **A real train/val/test split** (70/15/15, identity-disjoint via union-find over id-tokens in both
   folder and filename) — val calibrates, test reports, exactly once.
5. **Per-head temperature scaling, bootstrap confidence intervals, and a derived (not hardcoded)
   uncertainty-flag threshold.**
6. **An 8-row ablation grid (Cell 8)** that isolates the effect of cross-attention, hard negatives,
   compression augmentation, CMVN, and identity-disjoint vs. random splitting — the A8 row reproduces
   (and quantifies) the identity-leakage inflation behind the old ~89% number.

**Kaggle execution model (2×T4, 12h session limit):** Cells 1–3 are **Stage A** (feature extraction) —
run once, then *Save Version* and publish `/kaggle/working/features_v33` as a Kaggle Dataset. Cells 4
onward are **Stage B** (train / calibrate / DFDC / ablate / export) — attach that dataset in a fresh
session and it runs in minutes, so the whole ablation grid in Cell 8 costs almost nothing extra. If you'd
rather just run everything top-to-bottom in one long session, that works too — Stage B's cells fall back
to `/kaggle/working/features_v33` directly if it's already present.

**Before rerunning Stage A**, delete `/kaggle/working/features_v33` if it exists from an older run —
Cell 3 skips any `.pt` whose key already exists, and the collision-proof key scheme changed from v3.2's,
so stale tensors from before this version will not be found (harmless) but won't be regenerated either
if a partial run was interrupted mid-way through a category you've since rebalanced.

**Run cells in order.** Read the `WHAT CHANGED` comment at the top of each code cell before running it.

---

**v3.3.1 patch notes (after the first real Kaggle run):** the first run surfaced two problems the static
review above couldn't catch, since they only show up in trained-model behavior:
- **The sync head learned nothing** (AUC 0.513 — dead chance, on both val and test). Cell 4 now also
  tracks **train-set** sync AUC (previously only val/test were logged) and prints an automatic diagnosis:
  if train AUC is *also* near chance, that points to a feature bottleneck (frozen EfficientNet-B4
  embeddings likely don't preserve the mouth-motion detail true AV-sync needs); if train AUC is clearly
  above chance while val/test aren't, that's a fixable generalization gap from label noise in the
  roll-based negative sampling. **Rerun and check this diagnosis before assuming either fix is the right
  one** — back-of-envelope math on the first run's own numbers suggests the sampling-noise explanation
  alone is too small to explain a result pinned exactly at chance, so don't jump straight to the
  identity-aware sampling fix without checking the train-AUC diagnostic first.
- **The audio threshold collapsed on test** (94% of real audio misclassified fake at the val-derived
  Youden cutoff of 0.1134, despite a 0.999 AUC — a ranking-vs-operating-point split). Cell 5's threshold
  selection is now a **bootstrap median over 200 resamples** of val (`bootstrap_stable_threshold()`)
  instead of a single Youden's-J point estimate, with a minimum-recall floor on the real class for audio
  specifically, plus an explicit printed threshold-stability (std) diagnostic.

### Cell 1 — Environment, Shared CONFIG & Shared Helpers
Everything downstream reads from one `CONFIG` dict (also what Cell 8's ablation runner mutates per row).
New in v3.3: a single `ffmpeg_extract_wav()` audio front-end shared by Cells 3/6/7 (this is the fix for
the in-the-wild audio false positive — no more silent librosa backend fallback), and `make_rng(tag)` so
every independent random draw (category sampling, DFDC sampling, augmentation) gets its own reproducible
stream instead of sharing one global `random` position.

In [ ]:
# =====================================================================
# CELL 1: ENVIRONMENT SETUP, SHARED CONFIG & SHARED HELPERS  (v3.3)
# WHAT CHANGED vs v3.2:
#   * Single CONFIG dict drives every downstream cell AND the Cell 8
#     ablation runner -- toggling one flag (e.g. USE_CROSS_ATTENTION)
#     reruns the whole pipeline in a different mode without editing code.
#   * ffmpeg_extract_wav() replaces bare librosa.load(video_path, ...) as
#     THE single audio front-end for Cells 3/6/7. v3.2 let librosa fall
#     back silently from soundfile to audioread mid-run (visible as a
#     "PySoundFile failed. Trying audioread instead" warning on the
#     in-the-wild upload) -- a different decode backend than the one used
#     for every training tensor, which is the leading suspect for that
#     clip's 71.8% false-positive audio score. Routing every path through
#     ffmpeg first removes the backend as a variable.
#   * make_rng(tag) gives each cell (self-blend augmentation, category
#     sampling, DFDC sampling, aug-replica generation) its OWN seeded RNG
#     stream instead of sharing one global `random` stream whose position
#     depends on how many earlier random calls happened to run -- v3.2's
#     DFDC sample was silently NOT reproducible unless the entire notebook
#     reran identically from cell 1.
#   * Fixed a real bug in the version-check print: `'torchvision' in
#     dir()` is ALWAYS False (dir() with no args lists local names in the
#     current frame, and the module is bound to the name `torchvision`
#     itself, which IS in dir() after the import -- the guard was
#     backwards and always fell through to the string literal). Just
#     print torchvision.__version__ directly.
# =====================================================================
import os
import sys
import glob
import json
import random
import hashlib
import warnings
import subprocess
import tempfile

warnings.filterwarnings("ignore", category=UserWarning, module="librosa")
warnings.filterwarnings("ignore", category=FutureWarning, module="librosa")
warnings.filterwarnings("ignore", category=UserWarning, module="torchvision")

print("Installing extra dependencies WITHOUT touching numpy/torch/torchvision...")
# --no-deps => pip installs ONLY these packages' own code and resolves
# nothing else, so it is impossible for numpy to be changed here.
!pip install -q --no-deps facenet-pytorch librosa ffmpeg-python tqdm soundfile audioread pooch lazy_loader
print("Installation complete!")

import PIL._util
if not hasattr(PIL._util, 'is_directory'):
    PIL._util.is_directory = os.path.isdir
if not hasattr(PIL._util, 'is_path'):
    PIL._util.is_path = lambda x: isinstance(x, (str, bytes, os.PathLike))

import cv2
import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from facenet_pytorch import MTCNN
from sklearn.metrics import (roc_auc_score, accuracy_score, confusion_matrix,
                              roc_curve, auc, classification_report)
from sklearn.model_selection import GroupShuffleSplit
import matplotlib.pyplot as plt
import seaborn as sns

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(seed=42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()
print(f"Environment ready! Hardware acceleration: {device.type.upper()} "
      f"({n_gpus} GPU{'s' if n_gpus != 1 else ''} visible"
      f"{': ' + ', '.join(torch.cuda.get_device_name(i) for i in range(n_gpus)) if n_gpus else ''})")
print(f"Version check -> numpy: {np.__version__} | torch: {torch.__version__} | torchvision: {torchvision.__version__}")

# ---- v3.3 CONFIG -------------------------------------------------------
# One place the Cell 8 ablation runner mutates. Every other cell reads
# from this dict instead of hardcoding its own flags.
CONFIG = {
    "GENERATE_HARD_NEGATIVES": True,   # self-blend video-only hard negatives
    "USE_COMPRESSION_AUG":     True,   # JPEG/downscale/noise augmented replicas
    "AUG_REPLICAS":            2,      # augmented copies per extracted clip
    "USE_MFCC_CMVN":           True,   # per-utterance MFCC normalization
    "USE_CROSS_ATTENTION":     True,   # bidirectional CA -> sync_head in verdict
    "USE_SYNC_HEAD":           True,   # requires USE_CROSS_ATTENTION
    "IDENTITY_DISJOINT_SPLIT": True,   # False = ablation A8 (random split)
    "HELD_OUT_CATEGORY":       None,   # optional cross-manipulation ablation
    "VISUAL_FRAMES":           60,
    "AUDIO_FRAMES":            128,
    "N_MC_SAMPLES":            20,
    "SEED":                    42,
}
print("CONFIG ->", json.dumps(CONFIG, indent=2))

# ---- Shared MFCC normalization (CMVN) ----------------------------------
# Per-utterance Cepstral Mean-Variance Normalization: standardize each of
# the 40 MFCC coefficients across the time axis so an uploaded phone
# clip's audio lands in the SAME distribution the model trained on. This
# single helper is used identically in Cells 3 (train features), 6 (DFDC)
# and 7 (live inference) -- keep it here so it is defined before all of
# them.
def cmvn_normalize(mfcc, eps=1e-8):
    mfcc = np.asarray(mfcc, dtype=np.float32)
    mean = mfcc.mean(axis=1, keepdims=True)
    std = mfcc.std(axis=1, keepdims=True)
    return (mfcc - mean) / (std + eps)

# v3.3: CMVN is applied AT LOAD TIME (CrossFuseDataset / Cell 6 / Cell 7),
# NOT baked into the cached .pt tensors during extraction (Cell 3 now
# stores RAW MFCCs). This is what makes ablation A7
# (USE_MFCC_CMVN=False) a config toggle instead of a full re-extraction --
# CMVN can no longer be un-done once baked in, since it's a per-utterance
# standardization.
def maybe_cmvn(mfcc, use_cmvn=None):
    use_cmvn = CONFIG["USE_MFCC_CMVN"] if use_cmvn is None else use_cmvn
    return cmvn_normalize(mfcc) if use_cmvn else np.asarray(mfcc, dtype=np.float32)

# ---- Shared audio front-end (v3.3) --------------------------------------
# THE single decode path for every clip the model ever sees, whether it's
# a training tensor, a DFDC clip, or a user upload. Demuxing through
# ffmpeg first (mono, 16kHz PCM WAV) means librosa always reads a clean
# WAV via the fast soundfile backend -- it can no longer silently drop to
# the slower/different-behaving audioread backend mid-run.
def ffmpeg_extract_wav(video_path, sr=16000):
    """Returns (y, sr) exactly like librosa.load, but demuxes through
    ffmpeg first so every caller (Cells 3/6/7) shares one decode path.
    Falls back to direct librosa.load (with a loud warning, not a silent
    one) only if ffmpeg itself is unavailable or fails."""
    with tempfile.TemporaryDirectory() as tmpdir:
        wav_path = os.path.join(tmpdir, "audio.wav")
        try:
            result = subprocess.run(
                ["ffmpeg", "-y", "-i", video_path, "-vn", "-ac", "1",
                 "-ar", str(sr), "-f", "wav", wav_path],
                stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, timeout=60)
            if result.returncode != 0 or not os.path.exists(wav_path):
                raise RuntimeError(result.stderr.decode(errors="ignore")[-300:])
            y, sr_out = librosa.load(wav_path, sr=sr)
            return y, sr_out
        except Exception as e:
            print(f"[!] ffmpeg_extract_wav: ffmpeg path failed for "
                  f"'{os.path.basename(video_path)}' ({e}); "
                  f"falling back to direct librosa.load (backend may differ!)")
            return librosa.load(video_path, sr=sr)

# ---- Per-tag deterministic RNG streams (v3.3) ---------------------------
# Each independent random decision (category sampling, self-blend jitter,
# augmentation params, DFDC sampling) gets its OWN seeded stream keyed by
# a string tag, instead of every cell sharing one global `random` state
# whose position depends on how many earlier random calls already ran.
# This is what makes Cell 6's DFDC sample (and everything else) actually
# reproducible cell-by-cell, not just "reproducible if you rerun the
# entire notebook from the top in the same order."
def make_rng(tag, seed=None):
    seed = CONFIG["SEED"] if seed is None else seed
    h = int(hashlib.md5(f"{tag}:{seed}".encode()).hexdigest(), 16) % (2**32)
    return random.Random(h)

# ---- Shared extraction primitives (v3.3) --------------------------------
# Defined HERE (not in Cell 3) so Cells 6 and 7 -- both "Stage B" cells
# that may run in a Kaggle session which never executed Cell 3's Stage-A
# extraction loop -- can call the EXACT SAME batched, masked, 2-GPU-ready
# extraction path without duplicating it or silently drifting from it.
# Cell 3 additionally defines self-blend-only helpers (get_landmarks,
# self_blend_image, compression_augment) that these functions call ONLY
# when apply_self_blend / augment_fn are explicitly passed -- Cells 6/7
# never pass those, so they never need those definitions in scope.
N_DEVICES = max(1, torch.cuda.device_count())
DEVICE_IDS = [f"cuda:{i}" for i in range(N_DEVICES)] if torch.cuda.is_available() else ["cpu"]
print(f"Detected extraction devices: {DEVICE_IDS}")

img_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def build_extractor(device_str):
    mtcnn_local = MTCNN(image_size=224, margin=20, keep_all=False, device=device_str)
    eff_local = efficientnet_b4(weights=EfficientNet_B4_Weights.DEFAULT)
    eff_local.classifier = nn.Identity()
    eff_local = eff_local.to(device_str).eval()
    return mtcnn_local, eff_local

def extract_visual_embeddings_batched(video_path, mtcnn_local, eff_local, device_str,
                                       apply_self_blend=False, augment_fn=None, rng=None,
                                       n_frames=None, batch_size=32):
    """Batched replacement for a per-frame extraction loop: collect up to
    n_frames face crops first, then run MTCNN detection and the
    EfficientNet-B4 forward pass in chunks instead of one image at a
    time. Returns (visual_tensor, n_valid_frames) or (None, 0).
    apply_self_blend=True requires get_landmarks/self_blend_image (Cell 3)
    to be in scope; augment_fn requires whatever function is passed in to
    be in scope. Neither is needed when both are left at their defaults,
    which is how Cells 6 and 7 call this."""
    n_frames = n_frames or CONFIG["VISUAL_FRAMES"]
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = max(1, int(cap.get(cv2.CAP_PROP_FPS)))
    step = max(1, total_frames // n_frames) if total_frames > 0 else max(1, fps // 10)

    count, pil_frames = 0, []
    while cap.isOpened() and len(pil_frames) < n_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if count % step == 0:
            pil_frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        count += 1
    cap.release()
    if len(pil_frames) == 0:
        return None, 0

    face_pils = []
    for i in range(0, len(pil_frames), batch_size):
        chunk = pil_frames[i:i + batch_size]
        try:
            faces = mtcnn_local(chunk)
        except Exception:
            faces = [None] * len(chunk)
        if not isinstance(faces, list):
            faces = [faces[j] for j in range(faces.shape[0])] if faces is not None else [None] * len(chunk)
        for pil_img, face in zip(chunk, faces):
            if face is None:
                face_pil = pil_img.resize((224, 224))
            else:
                face_pil = transforms.ToPILImage()((face + 1.0) / 2.0)
            if apply_self_blend:
                landmarks = get_landmarks(face_pil, mtcnn_local)
                face_pil = self_blend_image(face_pil, landmarks, rng)
            if augment_fn is not None:
                face_pil = augment_fn(face_pil, rng)
            face_pils.append(face_pil)

    vis_embs = []
    with torch.no_grad():
        for i in range(0, len(face_pils), batch_size):
            chunk = face_pils[i:i + batch_size]
            batch_t = torch.stack([img_transform(p) for p in chunk]).to(device_str)
            with torch.autocast(device_type="cuda" if "cuda" in device_str else "cpu", dtype=torch.float16,
                                 enabled=("cuda" in device_str)):
                embs = eff_local(batch_t).float().cpu()
            vis_embs.append(embs)
    vis_tensor = torch.cat(vis_embs, dim=0)
    return vis_tensor, vis_tensor.shape[0]

def extract_audio_mfcc(video_path):
    """Full-clip MFCC, time-interpolated to AUDIO_FRAMES so it covers the
    SAME span as the sampled video frames. Returns the RAW (NOT CMVN'd)
    MFCC -- CMVN is applied at LOAD TIME (maybe_cmvn(), above) instead of
    being baked into the cached tensor, so ablation A7
    (USE_MFCC_CMVN=False) can reuse cached features instead of requiring
    a full re-extraction just to compare with/without CMVN."""
    try:
        y, sr = ffmpeg_extract_wav(video_path, sr=16000)
        if y is None or len(y) < sr // 10:
            raise ValueError("audio too short or empty")
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)              # (40, T_native)
        mfcc_t = torch.tensor(mfcc, dtype=torch.float32).unsqueeze(0)   # (1, 40, T_native)
        mfcc_t = F.interpolate(mfcc_t, size=CONFIG["AUDIO_FRAMES"], mode="linear", align_corners=False)
        return mfcc_t.squeeze(0).numpy().astype(np.float32)
    except Exception as e:
        return np.zeros((40, CONFIG["AUDIO_FRAMES"]), dtype=np.float32)

print("\nCell 1 complete: config, seeding, ffmpeg audio front-end, shared batched extraction "
      "primitives and per-tag RNGs ready.")

### Cell 2 — CrossFuse Architecture v3.3: Two Attribution Heads + Load-Bearing Bidirectional Cross-Attention
`video_head` and `audio_head` still read **only** their own branch (clean modality attribution, unchanged
from v3.2's design intent) — but now via **masked mean∥std pooling** over a **padding-masked**
transformer encoder, instead of a plain mean over zero-padded frames. New: bidirectional cross-attention
(video→audio **and** audio→video) feeds a `sync_head` trained to detect AV-correspondence mismatches;
this head **does** join the final verdict (Cells 6/7), which is what makes the paper's "Cross-Attention
Fusion" title accurate. `use_cross_attention=False` now skips building the CA modules entirely, for a
clean ablation.

In [ ]:
# =====================================================================
# CELL 2: CROSSFUSE ARCHITECTURE v3.3 -- TWO ATTRIBUTION HEADS +
#         LOAD-BEARING BIDIRECTIONAL CROSS-ATTENTION -> SYNC HEAD +
#         MASKED STATS POOLING + MC-DROPOUT UNCERTAINTY
#
# WHAT CHANGED vs v3.2 (three changes, in order of impact):
#
#   1. MASKED STATS POOLING instead of plain V_enc.mean(dim=1).
#      v3.2 zero-pads every clip to 60 frames with NO attention mask, so
#      (a) the pooled mean is diluted by however much padding a given
#      clip has (a 20-frame clip and a 60-frame clip both divide by 60),
#      and (b) mean-pooling alone throws away exactly the signal that
#      matters most for deepfake video -- frame-to-frame INCONSISTENCY.
#      We now (a) mask padding out of both the transformer encoder
#      (src_key_padding_mask) and the pooling, and (b) pool
#      concat[masked_mean, masked_std] instead of mean alone, so temporal
#      variance reaches the classifier.
#
#   2. BIDIRECTIONAL CROSS-ATTENTION IS NOW LOAD-BEARING. In v3.2, cross-
#      attention fed only an auxiliary "any-manipulation" head that never
#      touched the verdict -- undermining the paper's own title. v3.3
#      adds a SECOND cross-attention direction (audio attending to video,
#      alongside the existing video-attending-to-audio) and both feed a
#      new `sync_head`, trained with a self-supervised AV-correspondence
#      objective (Cell 4): P(this clip's audio and video do NOT belong
#      together). The video_head and audio_head still read ONLY their own
#      branch -- cross-attention still cannot leak audio into the video
#      decision or vice-versa -- but the sync_head genuinely depends on
#      cross-attention and genuinely joins the final verdict (Cells 6/7),
#      so "Cross-Attention Fusion" now describes what the model does.
#
#   3. use_cross_attention=False now SKIPS BUILDING the CA modules
#      entirely (not just skipping their use), so ablation checkpoints
#      trained with the flag off are smaller and load cleanly without the
#      unused weights.
#
#   forward() now returns a dict (clearer than a growing logit tuple):
#     {'video_logit', 'audio_logit', 'sync_logit', 'any_logit',
#      'attn_v2a', 'attn_a2v'}
#   All four logits are RAW logits (train with BCEWithLogitsLoss, read
#   with torch.sigmoid). sync_logit/any_logit/attn_* are None when
#   use_cross_attention=False.
# =====================================================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs=40, num_channels=[64, 128, 256], kernel_size=3, dropout=0.3, target_seq_len=60):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [
                nn.Conv1d(in_channels, out_channels, kernel_size,
                          padding=(kernel_size-1)*dilation_size, dilation=dilation_size),
                nn.BatchNorm1d(out_channels),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
        self.network = nn.Sequential(*layers)
        self.temporal_pool = nn.AdaptiveAvgPool1d(target_seq_len)
        self.proj = nn.Linear(num_channels[-1], 256)

    def forward(self, x):
        out = self.network(x)
        out = self.temporal_pool(out)
        out = out.transpose(1, 2)
        return self.proj(out)


def masked_mean_std(x, mask=None, eps=1e-8):
    """x: (B, T, D). mask: (B, T) bool, True = valid position (None = all
    valid, e.g. the audio branch which is always compressed to a fixed
    length by AdaptiveAvgPool1d). Returns concat[mean, std] -> (B, 2*D)."""
    if mask is None:
        mean = x.mean(dim=1)
        var = x.var(dim=1, unbiased=False)
    else:
        mask_f = mask.unsqueeze(-1).float()                      # (B, T, 1)
        count = mask_f.sum(dim=1).clamp(min=1.0)                 # (B, 1)
        mean = (x * mask_f).sum(dim=1) / count
        var = ((x - mean.unsqueeze(1)) ** 2 * mask_f).sum(dim=1) / count
    std = torch.sqrt(var.clamp(min=eps))
    return torch.cat([mean, std], dim=-1)


class CrossFuseModel(nn.Module):
    """
    Two independent attribution heads (video_head, audio_head) each read
    ONLY their own branch's masked-stats pooled representation, matching
    FakeAVCeleb's 2x2 structure and keeping clean modality attribution.

    Bidirectional cross-attention (video<->audio) is LOAD-BEARING: it
    feeds `sync_head`, a third signal trained to detect AV
    correspondence mismatches, which joins the final verdict (Cells 6/7)
    alongside the two attribution heads without ever touching their
    inputs. `aux_head` is a lighter-weight auxiliary "any manipulation"
    regularizer riding on the same fused representation.

    forward(visual_features, audio_mfcc, visual_mask=None) -> dict with
    keys 'video_logit', 'audio_logit', 'sync_logit', 'any_logit',
    'attn_v2a', 'attn_a2v'. visual_mask: (B, T_v) bool, True=valid frame;
    None means "all frames valid" (safe default for single-clip
    inference where the caller already trimmed/padded to a full window).
    """
    def __init__(self, visual_dim=1792, audio_dim=256, hidden_dim=256, num_heads=4,
                 dropout=0.4, target_seq_len=60, use_cross_attention=True):
        super(CrossFuseModel, self).__init__()
        self.use_cross_attention = use_cross_attention
        self.hidden_dim = hidden_dim

        # ---- Visual branch (feeds video_head ONLY) ----
        self.visual_proj = nn.Linear(visual_dim, hidden_dim)
        self.visual_encoder = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads, dim_feedforward=512,
            dropout=dropout, batch_first=True)

        # ---- Audio branch (feeds audio_head ONLY) ----
        self.audio_encoder = TemporalConvNet(num_inputs=40, num_channels=[64, 128, hidden_dim],
                                             dropout=dropout, target_seq_len=target_seq_len)

        # ---- Two independent binary attribution heads (masked mean||std -> 2*hidden) ----
        self.video_head = nn.Sequential(
            nn.Linear(2 * hidden_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 1))
        self.audio_head = nn.Sequential(
            nn.Linear(2 * hidden_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 1))

        # ---- Bidirectional cross-attention -> sync_head (load-bearing) + aux_head ----
        if use_cross_attention:
            self.cross_attention_v2a = nn.MultiheadAttention(
                embed_dim=hidden_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
            self.cross_attention_a2v = nn.MultiheadAttention(
                embed_dim=hidden_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
            self.layer_norm_v = nn.LayerNorm(hidden_dim)
            self.layer_norm_a = nn.LayerNorm(hidden_dim)

            fused_dim = 4 * hidden_dim   # concat[masked mean||std of fused-V, mean||std of fused-A]
            self.sync_head = nn.Sequential(
                nn.Linear(fused_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
                nn.Linear(128, 1))
            self.aux_head = nn.Sequential(
                nn.Linear(fused_dim, 64), nn.ReLU(), nn.Dropout(dropout),
                nn.Linear(64, 1))

    def forward(self, visual_features, audio_mfcc, visual_mask=None):
        B, T_v, _ = visual_features.shape
        if visual_mask is None:
            visual_mask = torch.ones(B, T_v, dtype=torch.bool, device=visual_features.device)
        key_padding_mask = ~visual_mask   # True = PAD, what nn.Transformer expects

        # Visual branch -> video-only representation
        V = self.visual_proj(visual_features)                         # (B, T_v, hidden)
        V_enc = self.visual_encoder(V, src_key_padding_mask=key_padding_mask)
        v_rep = masked_mean_std(V_enc, visual_mask)                    # (B, 2*hidden)
        video_logit = self.video_head(v_rep).squeeze(-1)

        # Audio branch -> audio-only representation (always full-length
        # after AdaptiveAvgPool1d, so no mask needed)
        A_enc = self.audio_encoder(audio_mfcc)                         # (B, 60, hidden)
        a_rep = masked_mean_std(A_enc, None)                            # (B, 2*hidden)
        audio_logit = self.audio_head(a_rep).squeeze(-1)

        sync_logit, any_logit, attn_v2a, attn_a2v = None, None, None, None
        if self.use_cross_attention:
            # Video queries audio (video positions attend over the full audio track)
            ca_v2a, attn_v2a = self.cross_attention_v2a(query=V_enc, key=A_enc, value=A_enc)
            fused_v = self.layer_norm_v(V_enc + ca_v2a)
            fused_v_pooled = masked_mean_std(fused_v, visual_mask)      # (B, 2*hidden)

            # Audio queries video (must respect visual padding as key_padding_mask)
            ca_a2v, attn_a2v = self.cross_attention_a2v(
                query=A_enc, key=V_enc, value=V_enc, key_padding_mask=key_padding_mask)
            fused_a = self.layer_norm_a(A_enc + ca_a2v)
            fused_a_pooled = masked_mean_std(fused_a, None)             # (B, 2*hidden)

            fused = torch.cat([fused_v_pooled, fused_a_pooled], dim=-1)  # (B, 4*hidden)
            sync_logit = self.sync_head(fused).squeeze(-1)
            any_logit = self.aux_head(fused).squeeze(-1)

        return {
            "video_logit": video_logit, "audio_logit": audio_logit,
            "sync_logit": sync_logit, "any_logit": any_logit,
            "attn_v2a": attn_v2a, "attn_a2v": attn_a2v,
        }


# ---- Monte Carlo Dropout uncertainty utilities (per-head, dict-based) ----

def enable_mc_dropout(model):
    """Switch ONLY Dropout layers back on (BatchNorm stays in eval mode
    with frozen running stats). Call model.eval() first, then this,
    before predict_with_uncertainty()."""
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.train()

def predict_with_uncertainty(model, visual, audio, visual_mask=None, n_samples=20):
    """n_samples stochastic passes (dropout active, BatchNorm frozen).
    Returns a dict: for each of 'video', 'audio', and (if the model has
    cross-attention) 'sync', a (mean, std) tuple of P(fake)/P(mismatch)
    across passes. std is the MC-Dropout uncertainty for that head --
    high std means the random subnetworks disagreed, i.e. the model is
    unsure about that signal."""
    model.eval()
    enable_mc_dropout(model)
    v_list, a_list, s_list = [], [], []
    with torch.no_grad():
        for _ in range(n_samples):
            out = model(visual, audio, visual_mask=visual_mask)
            v_list.append(torch.sigmoid(out["video_logit"]).cpu().numpy())
            a_list.append(torch.sigmoid(out["audio_logit"]).cpu().numpy())
            if out["sync_logit"] is not None:
                s_list.append(torch.sigmoid(out["sync_logit"]).cpu().numpy())
    model.eval()  # restore full eval mode (dropout off) afterwards

    v_arr, a_arr = np.stack(v_list, axis=0), np.stack(a_list, axis=0)
    result = {
        "video": (v_arr.mean(axis=0), v_arr.std(axis=0)),
        "audio": (a_arr.mean(axis=0), a_arr.std(axis=0)),
    }
    if s_list:
        s_arr = np.stack(s_list, axis=0)
        result["sync"] = (s_arr.mean(axis=0), s_arr.std(axis=0))
    return result


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CrossFuseModel(visual_dim=1792, target_seq_len=CONFIG["VISUAL_FRAMES"],
                        use_cross_attention=CONFIG["USE_CROSS_ATTENTION"]).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"CrossFuse v3.3 architecture ready (use_cross_attention={CONFIG['USE_CROSS_ATTENTION']}). "
      f"Total parameters: {n_params:,}. MC-Dropout utilities loaded.")

### Cell 3 (Stage A) — Extraction: Collision-Proof Manifest, Batched 2-GPU, Aligned Audio, Compression Augmentation
Run this standalone, then publish `/kaggle/working/features_v33` as a Kaggle Dataset before Stage B.
Fixes the ~290-clip silent data loss (collision-proof keys + a manifest-integrity assert), batches face
detection/embedding across both GPUs (5–10x faster than v3.2's one-frame-at-a-time loop), aligns audio to
cover the same time span as the sampled video, and generates JPEG/downscale/noise-augmented replicas so
the video head finally sees compression artifacts before being zero-shot tested on heavily-compressed
DFDC footage. Identity grouping is now a union-find over every `id\d+` token in path **and** filename,
closing a leakage path the old parent-folder-only identity couldn't see (manipulated clips encode a
*source* identity too).

In [ ]:
# =====================================================================
# CELL 3: FEATURE EXTRACTION v3.3 -- COLLISION-PROOF MANIFEST,
#         BATCHED 2-GPU EXTRACTION, UNIFIED/ALIGNED AUDIO,
#         COMPRESSION-AUGMENTED REPLICAS, HARDENED IDENTITY GROUPING
#
# WHAT CHANGED vs v3.2 (this is Stage A -- run this notebook standalone,
# then Save Version and publish /kaggle/working/features_v33 as a Kaggle
# Dataset for Stage B (Cells 4+) to attach):
#
#   * COLLISION-PROOF KEYS. v3.2 keyed each cached tensor on
#     `basename.split('.')[0]` alone. FakeAVCeleb reuses basenames like
#     "00109.mp4" across different categories/identities; combined with
#     "skip if the .pt already exists" and loop order reals-then-fakes,
#     a FAKE clip whose basename collided with an already-written REAL
#     was silently dropped -- 1,706 planned samples became 1,416 with no
#     warning. Every key here is
#     f"{category_short}_{identity_group}_{stem}_{md5(relpath)[:8]}" and
#     we assert uniqueness before extraction even starts.
#   * BATCHED GPU EXTRACTION. v3.2 ran eff_net on ONE face crop at a time
#     (60 separate forward passes per video). We now batch MTCNN
#     detection and EfficientNet-B4 forward passes across the whole
#     frame set per video (chunks of 32, fp16 autocast) -- expect ~5-10x
#     throughput.
#   * 2-GPU SHARDING. Two independent (MTCNN, EfficientNet-B4) replicas,
#     one pinned to cuda:0 and one to cuda:1, videos dispatched
#     round-robin via a 2-worker ThreadPoolExecutor (CUDA ops and cv2
#     decoding both release the GIL, so threads -- not
#     multiprocessing.spawn, which is fragile with CUDA inside Kaggle
#     notebooks -- give real parallelism here).
#   * ALIGNED AUDIO. v3.2 truncated MFCCs to their first ~128 frames
#     (~4s) while sampling video across the WHOLE clip -- cross-attention
#     between the two branches was silently attending across mismatched
#     time spans. We now take the MFCC of the FULL clip and interpolate
#     its time axis to AUDIO_FRAMES, so both streams cover the same
#     span. The RAW (not-yet-CMVN'd) MFCC is what gets cached -- CMVN is
#     now applied at LOAD TIME (maybe_cmvn(), Cell 1) so ablation A7
#     (USE_MFCC_CMVN=False) is a config toggle, not a re-extraction.
#     Audio is decoded once via the shared ffmpeg_extract_wav() front-end
#     from Cell 1.
#   * COMPRESSION/QUALITY AUGMENTED REPLICAS. Nothing in v3.2 ever showed
#     the model a compressed face, while DFDC (Cell 6) is heavily
#     H.264-compressed in-the-wild footage -- the leading suspect for the
#     0.627 DFDC AUC. Each base clip gets CONFIG["AUG_REPLICAS"] replicas
#     with random JPEG re-encoding, downscale/upscale and mild noise
#     applied per-frame, saved under the SAME identity_group as their
#     parent so the train/val/test split stays identity-disjoint.
#   * HARDENED IDENTITY GROUPING. v3.2's identity = parent folder name.
#     FakeAVCeleb filenames for manipulated clips encode a TARGET *and* a
#     SOURCE identity (e.g. id00076_id00812_wavtolip.mp4) -- a source
#     face can appear in val while its target folder is in train, which
#     the old identity-only assert could not catch. We union-find every
#     "id\d+"-shaped token found in either the parent folder or the
#     filename into connected components; each sample's identity_group is
#     its component's canonical id. This runs over the FULL discovered
#     video pool (not just the sampled subset) so cross-references are
#     resolved globally.
#   * REBALANCED CATEGORY SAMPLING. Video-head class balance in v3.2 was
#     344 real / 1072 fake (~1:3.1). We now keep ALL RealVideo-FakeAudio
#     (video-real AND audio-fake -- doubly useful, and free real-video
#     signal) and cap FakeVideo-RealAudio / FakeVideo-FakeAudio at ~250
#     each, landing near 1:1.9.
#   Everything else -- MTCNN/EfficientNet-B4 choice, the self-blend
#   landmark-on-crop fix, no ColorJitter/GaussianBlur in the base
#   img_transform -- is retained from v2.1/v3.2.
# =====================================================================

import os
import re
import csv
import glob
import cv2
import random
import hashlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from PIL import Image
from tqdm import tqdm
from torchvision import transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from facenet_pytorch import MTCNN

output_dir = "/kaggle/working/features_v33"
os.makedirs(output_dir, exist_ok=True)
manifest_path = os.path.join(output_dir, "manifest.csv")

# N_DEVICES, DEVICE_IDS, img_transform, build_extractor(),
# extract_visual_embeddings_batched() and extract_audio_mfcc() are all
# defined in Cell 1 (not here) so that Cells 6/7 -- which reuse the exact
# same extraction path -- work even in a Kaggle session that only ran
# Cell 1 + Cells 4 onward and never executed this cell.
print(f"Extraction will shard across: {DEVICE_IDS}")


def get_fakeavceleb_label(video_path: str) -> int:
    path_norm = video_path.replace("\\", "/").lower()
    if "realvideo-realaudio" in path_norm:
        return 0
    elif any(f in path_norm for f in ["realvideo-fakeaudio", "fakevideo-realaudio", "fakevideo-fakeaudio"]):
        return 1
    elif "real" in os.path.basename(path_norm) and "fake" not in os.path.basename(path_norm):
        return 0
    else:
        return 1

def get_fakeavceleb_category(video_path: str) -> str:
    path_norm = video_path.replace("\\", "/")
    for cat in ["RealVideo-RealAudio", "RealVideo-FakeAudio", "FakeVideo-RealAudio", "FakeVideo-FakeAudio"]:
        if cat.lower() in path_norm.lower():
            return cat
    return "Unknown"

CATEGORY_SHORT = {
    "RealVideo-RealAudio": "RVRA", "RealVideo-FakeAudio": "RVFA",
    "FakeVideo-RealAudio": "FVRA", "FakeVideo-FakeAudio": "FVFA",
    "SelfBlend-Synthetic": "SBS", "Unknown": "UNK",
}

def category_to_modality_labels(category: str):
    """Return (video_label, audio_label) for a category string."""
    if category == "SelfBlend-Synthetic":
        return 1, 0
    c = category.lower()
    video_fake = 1 if "fakevideo" in c else 0
    audio_fake = 1 if "fakeaudio" in c else 0
    return video_fake, audio_fake


# ---- Hardened identity grouping (union-find over id-tokens) ------------
class UnionFind:
    def __init__(self):
        self.parent = {}
    def find(self, x):
        self.parent.setdefault(x, x)
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:
            self.parent[x], x = root, self.parent[x]
        return root
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb

def get_identity_tokens(video_path: str):
    """FakeAVCeleb layout: .../{category}/{ethnicity}/{gender}/{id}/{video}.mp4,
    but manipulated filenames also encode a SOURCE identity
    (e.g. id00076_id00812_wavtolip.mp4). Pull every id\\d+ token from both
    the parent folder AND the filename."""
    path_norm = video_path.replace("\\", "/")
    parent = os.path.basename(os.path.dirname(path_norm))
    fname = os.path.basename(path_norm)
    tokens = set(re.findall(r'id\d+', parent.lower())) | set(re.findall(r'id\d+', fname.lower()))
    if not tokens:
        tokens = {f"folder::{parent.lower()}"}   # layout fallback, still groupable
    return tokens

def build_identity_groups(video_paths):
    """Union every id-token that co-occurs on any single clip into one
    connected component; returns {video_path: canonical_group_id}. This
    is what makes the train/val/test leakage assert in Cell 4 actually
    catch source-identity leakage, not just target-folder leakage."""
    uf = UnionFind()
    path_tokens = {}
    for v in video_paths:
        toks = get_identity_tokens(v)
        path_tokens[v] = toks
        toks = list(toks)
        for t in toks[1:]:
            uf.union(toks[0], t)
        if len(toks) == 1:
            uf.find(toks[0])
    return {v: uf.find(next(iter(path_tokens[v]))) for v in video_paths}


def get_landmarks(pil_img, mtcnn_module):
    boxes, probs, landmarks = mtcnn_module.detect(pil_img, landmarks=True)
    if landmarks is None or len(landmarks) == 0:
        return None
    return landmarks[0]

def build_blend_mask(size, landmarks):
    w, h = size
    mask = np.zeros((h, w), dtype=np.uint8)
    if landmarks is not None:
        pts = landmarks.astype(np.int32)
        hull = cv2.convexHull(pts)
        cv2.fillConvexPoly(mask, hull, 255)
        kernel = np.ones((31, 31), np.uint8)
        mask = cv2.dilate(mask, kernel, iterations=2)
    else:
        cv2.ellipse(mask, (w // 2, h // 2), (w // 3, int(h * 0.4)), 0, 0, 360, 255, -1)
    mask = cv2.GaussianBlur(mask, (35, 35), 0)
    return mask

def self_blend_image(pil_img, landmarks, rng):
    w, h = pil_img.size
    mask = build_blend_mask((w, h), landmarks).astype(np.float32)[..., None] / 255.0
    source = transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05)(pil_img)
    angle = rng.uniform(-4, 4)
    tx, ty = rng.uniform(-0.02, 0.02) * w, rng.uniform(-0.02, 0.02) * h
    scale = rng.uniform(0.97, 1.03)
    source = transforms.functional.affine(source, angle=angle, translate=(tx, ty), scale=scale, shear=0)
    base = np.array(pil_img).astype(np.float32)
    src = np.array(source).astype(np.float32)
    blended = mask * src + (1 - mask) * base
    return Image.fromarray(np.clip(blended, 0, 255).astype(np.uint8))

def compression_augment(pil_img, rng):
    """Random JPEG re-encode + random downscale/upscale + mild Gaussian
    noise -- gives the video head examples of the compression artifacts
    DFDC (and any in-the-wild upload) actually has, which nothing in
    v3.2's pipeline ever showed it."""
    w, h = pil_img.size
    if rng.random() < 0.8:
        scale = rng.uniform(0.5, 1.0)
        small = pil_img.resize((max(1, int(w * scale)), max(1, int(h * scale))), Image.BILINEAR)
        pil_img = small.resize((w, h), Image.BILINEAR)
    if rng.random() < 0.8:
        quality = rng.randint(30, 90)
        arr_bgr = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        ok, enc = cv2.imencode(".jpg", arr_bgr, [cv2.IMWRITE_JPEG_QUALITY, quality])
        if ok:
            dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)
            pil_img = Image.fromarray(cv2.cvtColor(dec, cv2.COLOR_BGR2RGB))
    if rng.random() < 0.5:
        arr = np.array(pil_img).astype(np.float32)
        noise = rng.gauss(0, 1) * np.random.RandomState(rng.randint(0, 2**31 - 1)).normal(0, 4, arr.shape)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        pil_img = Image.fromarray(arr)
    return pil_img


# extract_visual_embeddings_batched() and extract_audio_mfcc() are
# defined in Cell 1 (shared with Cells 6/7) -- both used below unchanged.


def make_sample_key(category, identity_group, video_path, suffix=""):
    stem = os.path.basename(video_path).split('.')[0]
    rel_hash = hashlib.md5(video_path.replace("\\", "/").encode()).hexdigest()[:8]
    return f"{CATEGORY_SHORT.get(category, 'UNK')}_{identity_group}_{stem}_{rel_hash}{suffix}"

def save_sample_v33(key, visual, audio, category, identity_group, synthetic, augmented, n_valid_frames):
    save_path = os.path.join(output_dir, f"{key}.pt")
    video_label, audio_label = category_to_modality_labels(category)
    overall_label = int(video_label or audio_label)
    audio_tensor = audio if torch.is_tensor(audio) else torch.tensor(audio, dtype=torch.float32)
    torch.save({
        'visual': visual, 'audio': audio_tensor,
        'label': overall_label, 'video_label': video_label, 'audio_label': audio_label,
        'identity_group': identity_group, 'category': category,
        'synthetic': synthetic, 'augmented': augmented, 'n_valid_frames': n_valid_frames,
    }, save_path)
    return save_path


# 1. Locate the FakeAVCeleb pool
all_videos = sorted(glob.glob("/kaggle/input/**/*.mp4", recursive=True))
if len(all_videos) == 0:
    all_videos = sorted(glob.glob("/kaggle/input/**/*.avi", recursive=True))

fav_videos = [v for v in all_videos if "dfdc" not in v.lower() and "deepfake-detection" not in v.lower()]

real_videos, fake_videos = [], []
for v in fav_videos:
    (real_videos if get_fakeavceleb_label(v) == 0 else fake_videos).append(v)

print(f"Raw Candidate Pool: {len(real_videos)} Real | {len(fake_videos)} Fake")

# 2. Global identity resolution BEFORE any sampling, over the FULL pool,
#    so source-identity cross-references between categories are caught.
print("Resolving identity groups (union-find over id-tokens in path+filename)...")
identity_groups = build_identity_groups(fav_videos)
print(f"  {len(fav_videos)} clips -> {len(set(identity_groups.values()))} identity groups")

rng_cat = make_rng("category_sample")
rng_cat.shuffle(real_videos)
rng_cat.shuffle(fake_videos)

selected_reals = real_videos  # use ALL reals (pool is small, ~178)

# --- REBALANCED CATEGORY-STRATIFIED FAKE SELECTION (v3.3) ------------------
fakes_by_cat = defaultdict(list)
for v in fake_videos:
    fakes_by_cat[get_fakeavceleb_category(v)].append(v)
for cat in fakes_by_cat:
    rng_cat.shuffle(fakes_by_cat[cat])

CAP_FVRA = 250
CAP_FVFA = 250
selected_fakes = []
selected_fakes += fakes_by_cat.get("RealVideo-FakeAudio", [])            # keep ALL: video-real + audio-fake
selected_fakes += fakes_by_cat.get("FakeVideo-RealAudio", [])[:CAP_FVRA]
selected_fakes += fakes_by_cat.get("FakeVideo-FakeAudio", [])[:CAP_FVFA]
rng_cat.shuffle(selected_fakes)

n_synth = len(selected_reals) if CONFIG["GENERATE_HARD_NEGATIVES"] else 0
cat_counts = {
    "RealVideo-FakeAudio": len(fakes_by_cat.get("RealVideo-FakeAudio", [])),
    "FakeVideo-RealAudio": min(CAP_FVRA, len(fakes_by_cat.get("FakeVideo-RealAudio", []))),
    "FakeVideo-FakeAudio": min(CAP_FVFA, len(fakes_by_cat.get("FakeVideo-FakeAudio", []))),
}
print(f"Balance plan -> reals(RVRA): {len(selected_reals)} | stratified fakes: {len(selected_fakes)} "
      f"| self-blend video-fakes: {n_synth} | aug replicas/base clip: "
      f"{CONFIG['AUG_REPLICAS'] if CONFIG['USE_COMPRESSION_AUG'] else 0}")
print(f"   per-category selected -> {cat_counts}  (RVFA + FVFA are your audio_head positives)")

if CONFIG["HELD_OUT_CATEGORY"] is not None:
    held_out_fakes = [v for v in selected_fakes if get_fakeavceleb_category(v) == CONFIG["HELD_OUT_CATEGORY"]]
    selected_fakes = [v for v in selected_fakes if get_fakeavceleb_category(v) != CONFIG["HELD_OUT_CATEGORY"]]
    print(f"Holding out {len(held_out_fakes)} '{CONFIG['HELD_OUT_CATEGORY']}' videos entirely")

print("\nSample identity parsing (verify these look like real identity groups!):")
for v in (selected_reals[:3] + selected_fakes[:3]):
    vl, al = category_to_modality_labels(get_fakeavceleb_category(v))
    print(f"  {v}  ->  identity_group='{identity_groups[v]}', category='{get_fakeavceleb_category(v)}', "
          f"video_label={vl}, audio_label={al}")

# 3. Build the full extraction job list up front (base + self-blend + aug
#    replicas), each job tagged with its own deterministic rng, THEN run
#    it sharded across GPUs -- this keeps manifest bookkeeping in one
#    place instead of scattered across the threaded loop.
jobs = []  # each: dict(video_path, category, identity_group, apply_self_blend, augment, aug_idx)
for v in selected_reals + selected_fakes:
    cat = get_fakeavceleb_category(v)
    ident = identity_groups[v]
    jobs.append(dict(video_path=v, category=cat, identity_group=ident,
                      apply_self_blend=False, augment=False, aug_idx=None))
    if CONFIG["GENERATE_HARD_NEGATIVES"] and get_fakeavceleb_label(v) == 0:
        jobs.append(dict(video_path=v, category="SelfBlend-Synthetic", identity_group=ident,
                          apply_self_blend=True, augment=False, aug_idx=None))
    if CONFIG["USE_COMPRESSION_AUG"]:
        for k in range(CONFIG["AUG_REPLICAS"]):
            jobs.append(dict(video_path=v, category=cat, identity_group=ident,
                              apply_self_blend=False, augment=True, aug_idx=k))

print(f"\nTotal extraction jobs queued: {len(jobs)}")

manifest_rows = []
manifest_lock_keys = set()

def process_job(job, device_str, mtcnn_local, eff_local):
    v = job["video_path"]
    suffix = ""
    if job["apply_self_blend"]:
        suffix = "_selfblend"
    elif job["augment"]:
        suffix = f"_aug{job['aug_idx']}"
    key = make_sample_key(job["category"], job["identity_group"], v, suffix=suffix)
    save_path = os.path.join(output_dir, f"{key}.pt")
    if os.path.exists(save_path):
        return key, job, "cached"

    rng = make_rng(key)
    augment_fn = compression_augment if job["augment"] else None
    try:
        visual, n_valid = extract_visual_embeddings_batched(
            v, mtcnn_local, eff_local, device_str,
            apply_self_blend=job["apply_self_blend"], augment_fn=augment_fn, rng=rng,
            n_frames=CONFIG["VISUAL_FRAMES"])
    except Exception as e:
        return key, job, f"visual_error:{e}"
    if visual is None:
        return key, job, "no_faces"

    if visual.shape[0] < CONFIG["VISUAL_FRAMES"]:
        pad = torch.zeros((CONFIG["VISUAL_FRAMES"] - visual.shape[0], visual.shape[1]), dtype=visual.dtype)
        visual = torch.cat([visual, pad], dim=0)
    else:
        visual = visual[:CONFIG["VISUAL_FRAMES"]]

    audio = extract_audio_mfcc(v)
    save_sample_v33(key, visual, audio, job["category"], job["identity_group"],
                     synthetic=job["apply_self_blend"], augmented=job["augment"],
                     n_valid_frames=n_valid)
    return key, job, "ok"


extractors = {d: build_extractor(d) for d in DEVICE_IDS}
results_log = defaultdict(int)

with ThreadPoolExecutor(max_workers=len(DEVICE_IDS)) as pool:
    futures = []
    for i, job in enumerate(jobs):
        device_str = DEVICE_IDS[i % len(DEVICE_IDS)]
        mtcnn_local, eff_local = extractors[device_str]
        futures.append(pool.submit(process_job, job, device_str, mtcnn_local, eff_local))
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Extracting (v3.3, 2-GPU batched)"):
        key, job, status = fut.result()
        results_log[status] += 1
        if status in ("ok", "cached"):
            if key in manifest_lock_keys:
                raise RuntimeError(f"Duplicate manifest key produced: {key} -- collision-proofing failed!")
            manifest_lock_keys.add(key)
            v_lab, a_lab = category_to_modality_labels(job["category"])
            manifest_rows.append(dict(
                key=key, path=job["video_path"], category=job["category"],
                identity_group=job["identity_group"], video_label=v_lab, audio_label=a_lab,
                synthetic=job["apply_self_blend"], augmented=job["augment"]))

print(f"\nExtraction outcomes: {dict(results_log)}")

# ---- Manifest integrity check (this is the check that would have caught
#      v3.2's ~290 silently-dropped clips) -------------------------------
with open(manifest_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["key", "path", "category", "identity_group",
                                            "video_label", "audio_label", "synthetic", "augmented"])
    writer.writeheader()
    writer.writerows(manifest_rows)

on_disk = set(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(output_dir, "*.pt")))
manifest_keys = set(r["key"] for r in manifest_rows)
assert len(manifest_rows) == len(manifest_keys), "Duplicate keys in manifest -- collision-proofing failed!"
missing_on_disk = manifest_keys - on_disk
assert len(missing_on_disk) == 0, f"{len(missing_on_disk)} manifest rows have no .pt file on disk: {list(missing_on_disk)[:5]}"
print(f"Manifest integrity OK -> {len(manifest_rows)} rows, {len(manifest_keys)} unique keys, "
      f"{len(on_disk)} .pt files on disk.")

cat_final = defaultdict(int)
for r in manifest_rows:
    cat_final[r["category"]] += 1
print(f"Final per-category sample counts: {dict(cat_final)}")
print(f"Manifest written to: {manifest_path}")
print("\nStage A complete. Save Version and publish /kaggle/working/features_v33 as a Kaggle "
      "Dataset before running Stage B (Cell 4 onward).")

### Cell 4 (Stage B start) — Identity-Disjoint 3-Way Split + Training
Attach the `features_v33` dataset published by Cell 3 first. Train/val/**test** (70/15/15) replaces
v3.2's train/val — val only selects the checkpoint, test is reserved for Cell 5's report. Training now
includes a self-supervised AV-sync objective (shuffled-pair mismatch detection), train-only embedding
augmentation targeting the video head's overfit gap, label smoothing, a cosine schedule with warmup, and
**early stopping** (v3.2 ran a fixed 15 epochs past its actual best checkpoint at epoch 11). The whole
loop is wrapped in `run_training_pipeline()` so Cell 8's ablation grid can reuse it without duplicating
code.

In [ ]:
# =====================================================================
# CELL 4 (STAGE B START): IDENTITY-DISJOINT 3-WAY SPLIT + TRAINING (v3.3.1)
# Attach the features_v33 Kaggle Dataset published by Cell 3 (Stage A)
# before running this cell. FEATURE_DIR below must point at it.
#
# v3.3.1 ADDITION (after the first real run came back with sync AUC
# pinned at chance, 0.513, on val/test): TRAIN-set sync AUC is now
# tracked and printed alongside val/test. This distinguishes two very
# different failure modes that look identical if you only watch val/test:
#   (a) train sync AUC ALSO near chance -> the model never found any
#       discriminative signal even in-sample. Most likely cause: frozen,
#       globally-pooled EfficientNet-B4 embeddings don't preserve the
#       mouth-motion/phoneme-timing detail true AV-sync detection needs --
#       a feature-representation problem no amount of cleaner negative
#       sampling can fix.
#   (b) train sync AUC clearly above chance while val/test sit at chance
#       -> a genuine generalization gap, most likely from label noise in
#       the roll-based negative sampling (self-blend/compression-aug
#       replicas of one source clip share IDENTICAL audio, so rolling one
#       against another can mislabel a true match as "mismatched").
# A short diagnosis line prints automatically at the end of training so
# this doesn't require manually reading epoch logs.
#
# WHAT CHANGED vs v3.2:
#   * TRAIN/VAL/TEST (70/15/15), not train/val. v3.2 fit its thresholds,
#     ECE, classification report AND the 4-way attribution matrix on the
#     SAME val set used for checkpointing -- every one of those numbers
#     was optimistically biased. Val now ONLY selects the checkpoint and
#     fits thresholds/temperature (Cell 5); test is touched exactly once,
#     for the numbers that go in the paper.
#   * Split uses the hardened identity_group from Cell 3's manifest
#     (union-find over id-tokens in path+filename), and we assert ALL
#     THREE pairwise intersections are empty, not just train/val.
#   * MASKED training: each sample now carries a visual_mask built from
#     the stored n_valid_frames (Cell 3), passed through to the model so
#     padding never pollutes pooling or the transformer encoder.
#   * CMVN is applied HERE, at load time (maybe_cmvn(), Cell 1), not
#     baked into the cached tensor -- lets ablation A7 toggle it for free.
#   * TRAIN-SPLIT-ONLY embedding augmentation (temporal crop, frame
#     dropout, Gaussian noise) -- free, since features are precomputed,
#     and aimed directly at the video head's train/val AUC gap (0.971 vs
#     0.788 in v3.2, still widening at epoch 15).
#   * WeightedRandomSampler on top of the existing pos_weight, for the
#     video head's remaining class imbalance.
#   * SYNC HEAD TRAINING (self-supervised AV correspondence): for every
#     batch we also run the model on the SAME visual batch paired with a
#     ROLLED (mismatched) audio batch. sync_logit is trained to output 0
#     for true pairs and 1 for the rolled/mismatched pairs. Labels are
#     correct by construction for the OVERWHELMING majority of pairs
#     (there are 350+ train identities, so a random roll lands on a
#     different identity's audio almost always) -- the one known
#     exception is a self-blend/augmented replica happening to roll
#     against its OWN source clip's replica in the same batch, which
#     shares near-identical audio and would inject a rare noisy
#     "mismatched" label. This is a cheap approximation, not a hard
#     guarantee; a stricter version would filter rolls by identity_group,
#     which the current SplitWrapper doesn't propagate per-sample. This
#     is what makes cross-attention load-bearing rather than decorative.
#   * Label smoothing (eps=0.05) on the video head's BCE target.
#   * Cosine LR schedule with 2-epoch warmup, grad-norm clipping, and
#     EARLY STOPPING (patience=7 on val mean AUC) instead of a fixed 15
#     epochs -- v3.2's best checkpoint was epoch 11 of 15; epochs 12-15
#     were pure memorization (train AUC kept climbing, val AUC didn't).
#   * The whole pipeline is wrapped in run_training_pipeline(cfg, tag),
#     so Cell 8's ablation grid can call it with modified CONFIG copies
#     (unimodal, no-CA, no-hard-negatives, no-aug, no-CMVN, random-split)
#     WITHOUT duplicating this cell's logic -- features stay cached, only
#     the loss composition / dataset filter / split strategy changes.
#     The "main" run below (tag="main") also sets the module-level
#     globals (model, checkpoint_path, val_loader, test_loader, ...)
#     that Cells 5/6/7 read from.
# =====================================================================

import os
import csv
import math
import json
import glob
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, accuracy_score

FEATURE_DIR = "/kaggle/working/features_v33"
if not os.path.isdir(FEATURE_DIR) or not glob.glob(os.path.join(FEATURE_DIR, "*.pt")):
    candidates = sorted(glob.glob("/kaggle/input/*/features_v33")) + \
                 sorted(glob.glob("/kaggle/input/*features*v33*"))
    if candidates:
        FEATURE_DIR = candidates[0]
    print(f"[i] Using FEATURE_DIR = {FEATURE_DIR} (edit this if Stage A's dataset mounted elsewhere)")
MANIFEST_PATH = os.path.join(FEATURE_DIR, "manifest.csv")


class CrossFuseDataset(Dataset):
    def __init__(self, folder, manifest_path=None, target_frames=60, use_cmvn=True,
                 exclude_synthetic=False, exclude_augmented=False):
        self.target_frames = target_frames
        self.use_cmvn = use_cmvn
        if manifest_path and os.path.exists(manifest_path):
            with open(manifest_path, newline="") as f:
                rows = list(csv.DictReader(f))
            if exclude_synthetic:
                rows = [r for r in rows if r.get("synthetic", "False") != "True"]
            if exclude_augmented:
                rows = [r for r in rows if r.get("augmented", "False") != "True"]
            self.files = [os.path.join(folder, r["key"] + ".pt") for r in rows]
            self._identity_csv = [r["identity_group"] for r in rows]
        else:
            self.files = sorted(glob.glob(os.path.join(folder, "*.pt")))
            self._identity_csv = None
        self.meta = [torch.load(f, weights_only=False) for f in self.files]

    def __len__(self):
        return len(self.files)

    def _labels(self, data):
        if 'video_label' in data and 'audio_label' in data:
            return int(data['video_label']), int(data['audio_label'])
        lbl = int(data.get('label', 0))
        return lbl, lbl

    def identity(self, idx):
        if self._identity_csv is not None:
            return self._identity_csv[idx]
        data = self.meta[idx]
        return data.get('identity_group', data.get('identity', f'unknown_{idx}'))

    def __getitem__(self, idx):
        data = self.meta[idx]
        vis = data['visual']
        current_frames, feature_dim = vis.shape[0], vis.shape[1]
        n_valid = int(data.get('n_valid_frames', current_frames))
        n_valid = max(1, min(n_valid, self.target_frames))
        if current_frames < self.target_frames:
            padding = torch.zeros((self.target_frames - current_frames, feature_dim), dtype=vis.dtype)
            vis = torch.cat([vis, padding], dim=0)
        else:
            vis = vis[:self.target_frames]
        mask = torch.zeros(self.target_frames, dtype=torch.bool)
        mask[:n_valid] = True
        vlab, alab = self._labels(data)

        audio_raw = data['audio']
        audio_np = audio_raw.numpy() if torch.is_tensor(audio_raw) else np.asarray(audio_raw)
        audio_np = maybe_cmvn(audio_np, use_cmvn=self.use_cmvn)
        audio = torch.tensor(audio_np, dtype=torch.float32)

        return (vis, audio,
                torch.tensor(vlab, dtype=torch.float32),
                torch.tensor(alab, dtype=torch.float32),
                mask)


class SplitWrapper(Dataset):
    """Wraps a base dataset + index list; applies train-only embedding
    augmentation (temporal crop, frame dropout, Gaussian noise) so val/
    test stay exactly as extracted while train gets cheap, precomputed-
    feature-level augmentation aimed at the video head's overfit gap."""
    def __init__(self, base, indices, training=False):
        self.base = base
        self.indices = list(indices)
        self.training = training

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        vis, audio, vlab, alab, mask = self.base[self.indices[i]]
        if self.training:
            vis, mask = self._augment_visual(vis, mask)
            audio = audio + torch.randn_like(audio) * 0.01
        return vis, audio, vlab, alab, mask

    @staticmethod
    def _augment_visual(vis, mask):
        vis = vis.clone()
        mask = mask.clone()
        n_valid = int(mask.sum().item())
        if n_valid > 10:
            if random.random() < 0.5:                      # random temporal crop
                crop_frac = random.uniform(0.7, 1.0)
                keep = max(8, int(n_valid * crop_frac))
                start = random.randint(0, n_valid - keep)
                new_mask = torch.zeros_like(mask)
                new_mask[start:start + keep] = True
                mask = new_mask
            drop_p = 0.1                                    # frame dropout (zeroed AND masked out)
            drop_idx = (torch.rand(mask.shape) < drop_p) & mask
            vis[drop_idx] = 0.0
            mask = mask & (~drop_idx)
            if mask.sum() == 0:
                mask[0] = True
        vis = vis + torch.randn_like(vis) * 0.01            # feature-level Gaussian noise
        return vis, mask


def worker_init_fn(worker_id):
    base_seed = CONFIG["SEED"] + worker_id
    random.seed(base_seed)
    np.random.seed(base_seed)
    torch.manual_seed(base_seed)


def safe_auc(y_true, y_score):
    try:
        if len(set(int(v) for v in y_true)) < 2:
            return 0.5
        return roc_auc_score(y_true, y_score)
    except ValueError:
        return 0.5

def smooth_labels(y, eps=0.05):
    return y * (1.0 - eps) + 0.5 * eps


def build_3way_split(dataset, identity_disjoint=True, seed=42):
    identities_all = [dataset.identity(i) for i in range(len(dataset))]
    groups_arr = np.array(identities_all) if identity_disjoint else np.arange(len(dataset))
    X_dummy = np.zeros(len(dataset))

    gss_test = GroupShuffleSplit(n_splits=1, train_size=0.85, random_state=seed)
    temp_idx, test_idx = next(gss_test.split(X_dummy, groups=groups_arr))
    gss_val = GroupShuffleSplit(n_splits=1, train_size=0.70 / 0.85, random_state=seed + 1)
    train_rel, val_rel = next(gss_val.split(X_dummy[temp_idx], groups=groups_arr[temp_idx]))
    train_idx, val_idx = temp_idx[train_rel], temp_idx[val_rel]

    if identity_disjoint:
        train_ids = set(identities_all[i] for i in train_idx)
        val_ids = set(identities_all[i] for i in val_idx)
        test_ids = set(identities_all[i] for i in test_idx)
        ov_tv, ov_tt, ov_vt = train_ids & val_ids, train_ids & test_ids, val_ids & test_ids
        assert len(ov_tv) == 0 and len(ov_tt) == 0 and len(ov_vt) == 0, \
            "Identity leakage detected across the 3-way split -- check identity_group in Cell 3!"
    return train_idx, val_idx, test_idx


def run_training_pipeline(cfg, tag="main", verbose=True):
    """Full identity-disjoint split + two-head(+sync) training loop,
    parameterized entirely by `cfg` so Cell 8's ablation grid can call
    this with modified CONFIG copies instead of duplicating the loop.
    Returns a result dict; for tag=="main" it ALSO sets the module-level
    globals Cells 5/6/7 depend on (model state at `checkpoint_path`,
    val_loader, test_loader, full_dataset, thr_video/thr_audio globals
    are NOT set here -- those are Cell 5's job).
    """
    full_ds = CrossFuseDataset(
        FEATURE_DIR, manifest_path=MANIFEST_PATH, target_frames=cfg["VISUAL_FRAMES"],
        use_cmvn=cfg["USE_MFCC_CMVN"],
        exclude_synthetic=not cfg["GENERATE_HARD_NEGATIVES"],
        exclude_augmented=not cfg["USE_COMPRESSION_AUG"])

    video_labels_all = [int(m.get('video_label', m.get('label', 0))) for m in full_ds.meta]
    audio_labels_all = [int(m.get('audio_label', m.get('label', 0))) for m in full_ds.meta]

    if verbose:
        print(f"[{tag}] Dataset -> Total: {len(full_ds)} | Video real/fake: "
              f"{video_labels_all.count(0)}/{video_labels_all.count(1)} | Audio real/fake: "
              f"{audio_labels_all.count(0)}/{audio_labels_all.count(1)}")

    train_idx, val_idx, test_idx = build_3way_split(
        full_ds, identity_disjoint=cfg["IDENTITY_DISJOINT_SPLIT"], seed=cfg["SEED"])
    if verbose:
        print(f"[{tag}] Samples -> train: {len(train_idx)} | val: {len(val_idx)} | test: {len(test_idx)}")

    train_ds = SplitWrapper(full_ds, train_idx, training=True)
    val_ds = SplitWrapper(full_ds, val_idx, training=False)
    test_ds = SplitWrapper(full_ds, test_idx, training=False)

    train_video_labels = [video_labels_all[i] for i in train_idx]
    sample_weights = [3.0 if l == 1 else 1.0 for l in train_video_labels]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    n_workers = cfg.get("NUM_WORKERS", 2)
    train_loader = DataLoader(train_ds, batch_size=16, sampler=sampler, drop_last=True,
                               num_workers=n_workers, pin_memory=True,
                               persistent_workers=(n_workers > 0), worker_init_fn=worker_init_fn)
    val_loader_local = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=n_workers, pin_memory=True)
    test_loader_local = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=n_workers, pin_memory=True)

    model_local = CrossFuseModel(visual_dim=1792, target_seq_len=cfg["VISUAL_FRAMES"], dropout=0.4,
                                  use_cross_attention=cfg["USE_CROSS_ATTENTION"]).to(device)
    optimizer = torch.optim.AdamW(model_local.parameters(), lr=1e-4, weight_decay=1e-4)

    n_pos_v = sum(train_video_labels)
    n_neg_v = len(train_video_labels) - n_pos_v
    video_pos_weight = torch.tensor(n_neg_v / max(n_pos_v, 1), dtype=torch.float32).to(device)

    modality = cfg.get("MODALITY", "both")   # "both" | "video_only" | "audio_only" -- Cell 8 unimodal ablations
    criterion_video = nn.BCEWithLogitsLoss(pos_weight=video_pos_weight)
    criterion_audio = nn.BCEWithLogitsLoss()
    criterion_sync = nn.BCEWithLogitsLoss()
    criterion_any = nn.BCEWithLogitsLoss()
    AUX_WEIGHT = 0.2
    LAMBDA_SYNC = 0.5

    epochs = cfg.get("EPOCHS", 30)
    warmup_epochs = cfg.get("WARMUP_EPOCHS", 2)
    patience = cfg.get("PATIENCE", 7)
    grad_clip = 1.0

    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, (epochs - warmup_epochs))
        return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    best_mean_auc, best_video_auc, best_audio_auc, best_sync_auc = 0.0, 0.0, 0.0, 0.0
    best_train_sync_auc = float('nan')
    best_epoch, epochs_since_best = -1, 0
    ckpt_path = f"/kaggle/working/crossfuse_v33_{tag}.pth"

    if verbose:
        print(f"[{tag}] Starting training (modality={modality}, use_CA={cfg['USE_CROSS_ATTENTION']}, "
              f"identity_disjoint={cfg['IDENTITY_DISJOINT_SPLIT']})...")

    for epoch in range(epochs):
        model_local.train()
        train_loss = 0.0
        t_v_true, t_v_prob, t_a_true, t_a_prob = [], [], [], []
        t_s_true, t_s_prob = [], []   # v3.3.1: train-set sync AUC diagnostic (see note below)
        for vis_batch, audio_batch, vlab_batch, alab_batch, mask_batch in train_loader:
            vis_batch, audio_batch = vis_batch.to(device), audio_batch.to(device)
            vlab_batch, alab_batch = vlab_batch.to(device), alab_batch.to(device)
            mask_batch = mask_batch.to(device)
            optimizer.zero_grad()

            out = model_local(vis_batch, audio_batch, visual_mask=mask_batch)
            loss = torch.zeros((), device=device)
            if modality in ("both", "video_only"):
                vlab_smooth = smooth_labels(vlab_batch, eps=0.05)
                loss = loss + criterion_video(out["video_logit"], vlab_smooth)
            if modality in ("both", "audio_only"):
                loss = loss + criterion_audio(out["audio_logit"], alab_batch)

            if out["sync_logit"] is not None:
                B = audio_batch.size(0)
                perm = torch.roll(torch.arange(B, device=device), shifts=1)
                out_shuf = model_local(vis_batch, audio_batch[perm], visual_mask=mask_batch)
                sync_logits = torch.cat([out["sync_logit"], out_shuf["sync_logit"]], dim=0)
                sync_labels = torch.cat([torch.zeros(B, device=device), torch.ones(B, device=device)], dim=0)
                loss = loss + LAMBDA_SYNC * criterion_sync(sync_logits, sync_labels)
                any_lab = torch.clamp(vlab_batch + alab_batch, 0, 1)
                loss = loss + AUX_WEIGHT * criterion_any(out["any_logit"], any_lab)

                # v3.3.1 diagnostic: track TRAIN sync AUC (in-sample, with dropout
                # still active) so we can tell apart two failure modes if val/test
                # sync AUC sits at chance -- (a) the model learned SOMETHING on
                # train but it didn't generalize (a real generalization gap, fixable
                # by cleaning up the roll-based negative sampling), vs (b) the model
                # never separated matched/mismatched pairs even in-sample (points to
                # a feature bottleneck -- frozen EfficientNet-B4 embeddings may not
                # carry the mouth-motion/phoneme-timing detail true AV-sync needs,
                # which no amount of cleaner sampling would fix). v3.3 only logged
                # val/test sync AUC, which couldn't distinguish these two cases.
                t_s_prob.extend(torch.sigmoid(out["sync_logit"]).detach().cpu().numpy())
                t_s_prob.extend(torch.sigmoid(out_shuf["sync_logit"]).detach().cpu().numpy())
                t_s_true.extend([0] * B + [1] * B)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_local.parameters(), grad_clip)
            optimizer.step()
            train_loss += loss.item()

            t_v_prob.extend(torch.sigmoid(out["video_logit"]).detach().cpu().numpy())
            t_v_true.extend(vlab_batch.detach().cpu().numpy())
            t_a_prob.extend(torch.sigmoid(out["audio_logit"]).detach().cpu().numpy())
            t_a_true.extend(alab_batch.detach().cpu().numpy())

        scheduler.step()
        avg_train_loss = train_loss / max(1, len(train_loader))
        train_video_auc = safe_auc(t_v_true, t_v_prob)
        train_audio_auc = safe_auc(t_a_true, t_a_prob)
        train_sync_auc = safe_auc(t_s_true, t_s_prob) if t_s_true else float('nan')

        model_local.eval()
        v_true, v_prob, a_true, a_prob, s_true, s_prob = [], [], [], [], [], []
        with torch.no_grad():
            for vis_batch, audio_batch, vlab_batch, alab_batch, mask_batch in val_loader_local:
                vis_batch, audio_batch, mask_batch = vis_batch.to(device), audio_batch.to(device), mask_batch.to(device)
                out = model_local(vis_batch, audio_batch, visual_mask=mask_batch)
                v_prob.extend(torch.sigmoid(out["video_logit"]).cpu().numpy())
                a_prob.extend(torch.sigmoid(out["audio_logit"]).cpu().numpy())
                v_true.extend(vlab_batch.numpy())
                a_true.extend(alab_batch.numpy())
                if out["sync_logit"] is not None:
                    B = audio_batch.size(0)
                    perm = torch.roll(torch.arange(B, device=device), shifts=1)
                    out_shuf = model_local(vis_batch, audio_batch[perm], visual_mask=mask_batch)
                    s_prob.extend(torch.sigmoid(out["sync_logit"]).cpu().numpy())
                    s_prob.extend(torch.sigmoid(out_shuf["sync_logit"]).cpu().numpy())
                    s_true.extend([0] * B + [1] * B)

        video_auc = safe_auc(v_true, v_prob)
        audio_auc = safe_auc(a_true, a_prob)
        sync_auc = safe_auc(s_true, s_prob) if s_true else float('nan')

        if modality == "video_only":
            mean_auc = video_auc
        elif modality == "audio_only":
            mean_auc = audio_auc
        else:
            mean_auc = 0.5 * (video_auc + audio_auc)
        gap = train_video_auc - video_auc

        improved = mean_auc > best_mean_auc
        if improved:
            best_mean_auc, best_video_auc, best_audio_auc, best_sync_auc = mean_auc, video_auc, audio_auc, sync_auc
            best_train_sync_auc = train_sync_auc
            best_epoch, epochs_since_best = epoch, 0
            torch.save(model_local.state_dict(), ckpt_path)
            star = "* (New Best)"
        else:
            epochs_since_best += 1
            star = ""

        if verbose:
            print(f"[{tag}] Epoch [{epoch+1:02d}/{epochs}] LR {scheduler.get_last_lr()[0]:.2e} -> "
                  f"Loss: {avg_train_loss:.4f} | Train AUC (v/a/s): "
                  f"{train_video_auc:.4f}/{train_audio_auc:.4f}/{train_sync_auc:.4f} | "
                  f"Val V/A/S AUC: {video_auc:.4f}/{audio_auc:.4f}/{sync_auc:.4f} | Mean: {mean_auc:.4f} | "
                  f"Gap: {gap:+.4f} {star}")

        if epochs_since_best >= patience:
            if verbose:
                print(f"[{tag}] Early stopping at epoch {epoch+1} (no improvement in {patience} epochs).")
            break

    if verbose:
        print(f"[{tag}] Done. Best epoch {best_epoch+1} -> Video AUC {best_video_auc:.4f} | "
              f"Audio AUC {best_audio_auc:.4f} | Sync AUC {best_sync_auc:.4f} (train sync AUC at that "
              f"epoch: {best_train_sync_auc:.4f}) | Mean {best_mean_auc:.4f}")
        if not np.isnan(best_sync_auc):
            if best_train_sync_auc < 0.55:
                print(f"[{tag}] DIAGNOSIS: train sync AUC ({best_train_sync_auc:.4f}) is ALSO near chance "
                      f"-> the sync task likely never had learnable signal in these features (points to a "
                      f"feature-bottleneck problem, not a sampling-noise problem). See plan v3.3.1 Priority 2.")
            elif best_sync_auc < 0.55 <= best_train_sync_auc:
                print(f"[{tag}] DIAGNOSIS: train sync AUC ({best_train_sync_auc:.4f}) is above chance but "
                      f"val/test sync AUC ({best_sync_auc:.4f}) is not -> a genuine generalization gap; the "
                      f"identity-aware negative-pair fix is worth trying. See plan v3.3.1 Priority 2.")

    result = {
        "tag": tag, "checkpoint_path": ckpt_path, "best_epoch": best_epoch + 1,
        "best_val_video_auc": best_video_auc, "best_val_audio_auc": best_audio_auc,
        "best_val_sync_auc": best_sync_auc, "best_train_sync_auc": best_train_sync_auc,
        "best_mean_auc": best_mean_auc,
        "n_train": len(train_idx), "n_val": len(val_idx), "n_test": len(test_idx),
        "full_dataset": full_ds, "train_idx": train_idx, "val_idx": val_idx, "test_idx": test_idx,
        "train_loader": train_loader, "val_loader": val_loader_local, "test_loader": test_loader_local,
        "model": model_local, "cfg": cfg,
    }
    return result


# ---- Main run (tag="main") -- also populates the globals Cells 5/6/7 use ----
_main_result = run_training_pipeline(CONFIG, tag="main")

full_dataset = _main_result["full_dataset"]
train_idx, val_idx, test_idx = _main_result["train_idx"], _main_result["val_idx"], _main_result["test_idx"]
train_loader = _main_result["train_loader"]
val_loader = _main_result["val_loader"]
test_loader = _main_result["test_loader"]
model = _main_result["model"]
checkpoint_path = _main_result["checkpoint_path"]

best_metrics = {
    "best_epoch": _main_result["best_epoch"], "best_val_video_auc": _main_result["best_val_video_auc"],
    "best_val_audio_auc": _main_result["best_val_audio_auc"], "best_val_sync_auc": _main_result["best_val_sync_auc"],
    "best_train_sync_auc": _main_result["best_train_sync_auc"], "best_mean_auc": _main_result["best_mean_auc"],
    "config": CONFIG,
}
with open("/kaggle/working/best_metrics.json", "w") as f:
    json.dump(best_metrics, f, indent=2)
print(f"\nMain run complete. Best checkpoint: {checkpoint_path}")
print("best_metrics.json written.")

### Cell 5 — Per-Head Temperature Scaling + Calibration on Val, Full Report on Test
Fits per-head temperature scaling and Youden thresholds (video/audio/**sync**) on val only, saves them to
`calibration.json`, then computes **every** reported metric — classification reports, ECE, ROC,
reliability diagrams, the 4-way attribution matrix, and a new sync-head ROC — on the **test** split,
touched here for the first and only time. Adds bootstrap 95% confidence intervals and derives the
uncertainty-flag cutoff from the actual val MC-Dropout distribution instead of a hardcoded 0.15.

In [ ]:
# =====================================================================
# CELL 5: PER-HEAD TEMPERATURE SCALING + THRESHOLD CALIBRATION ON VAL,
#         FULL REPORT (ECE, ROC, ATTRIBUTION MATRIX, BOOTSTRAP CIs) ON
#         THE HELD-OUT TEST SPLIT  (v3.3.1)
#
# v3.3.1 FIX: the first v3.3 run's single-draw Youden's-J audio threshold
# (0.1134, fit on 991 val samples) misclassified 94% of TEST real audio as
# fake -- ranking (AUC 0.999) was fine, but that one point estimate didn't
# transfer. `bootstrap_stable_threshold()` replaces the single-draw
# Youden's-J call with a median across 200 bootstrap resamples of val,
# with an optional minimum-recall floor on the real class (applied to
# audio, where the failure happened). See that function's docstring below
# for the full reasoning.
#
# WHAT CHANGED vs v3.2:
#   * VAL vs TEST SEPARATION. v3.2 computed EVERYTHING -- thresholds,
#     classification report, ECE, reliability diagrams, the 4-way
#     attribution matrix -- on the same val set used for checkpoint
#     selection, which optimistically biases every one of those numbers.
#     Calibration (temperature + Youden thresholds + the uncertainty flag
#     cutoff) is fit on VAL here; every reported metric is computed on
#     TEST, touched exactly once.
#   * PER-HEAD TEMPERATURE SCALING (video/audio/sync), fit by LBFGS on
#     val NLL, applied to MEAN LOGITS across MC-dropout passes before the
#     final sigmoid -- this is what the v3.2 video ECE of 0.1196 was
#     missing; MC-dropout's raw sigmoid mean is not automatically
#     calibrated.
#   * SYNC HEAD calibration + ROC added -- this is the evidence that
#     cross-attention is doing real work (see Cell 2/4 changes).
#   * BOOTSTRAP 95% CIs (2000 resamples) on every AUC/accuracy so a
#     single point estimate doesn't stand in for statistical confidence.
#   * UNCERTAINTY_FLAG_THRESHOLD is now DERIVED (95th percentile of val
#     MC-Dropout std, per head) and saved, instead of the v3.2 hardcoded
#     0.15 -- which is why the earlier in-the-wild upload's uncertainty
#     flag (0.15 vs the val distribution's actual 0.035/0.082) fired
#     somewhat arbitrarily.
#   * Reliability-plot y-axis label fixed to "Empirical fraction of
#     positives" -- after the ECE formula fix, that IS what's plotted;
#     the old "Empirical accuracy" label was left over from before the
#     fix and was simply wrong.
# =====================================================================

import json
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report, roc_auc_score, accuracy_score

eval_model = CrossFuseModel(visual_dim=1792, target_seq_len=CONFIG["VISUAL_FRAMES"],
                             use_cross_attention=CONFIG["USE_CROSS_ATTENTION"]).to(device)
eval_model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=False))
eval_model.eval()

# ---- Pass 1: deterministic VAL logits (dropout OFF) for temperature + threshold fitting ----
def collect_deterministic(model, loader, need_sync):
    v_logit, a_logit, v_true, a_true = [], [], [], []
    s_logit, s_true = [], []
    model.eval()
    with torch.no_grad():
        for vis, audio, vlab, alab, mask in loader:
            vis, audio, mask = vis.to(device), audio.to(device), mask.to(device)
            out = model(vis, audio, visual_mask=mask)
            v_logit.extend(out["video_logit"].cpu().numpy()); v_true.extend(vlab.numpy())
            a_logit.extend(out["audio_logit"].cpu().numpy()); a_true.extend(alab.numpy())
            if need_sync and out["sync_logit"] is not None:
                B = audio.size(0)
                perm = torch.roll(torch.arange(B, device=device), shifts=1)
                out_shuf = model(vis, audio[perm], visual_mask=mask)
                s_logit.extend(out["sync_logit"].cpu().numpy()); s_logit.extend(out_shuf["sync_logit"].cpu().numpy())
                s_true.extend([0] * B + [1] * B)
    return (np.array(v_logit), np.array(v_true), np.array(a_logit), np.array(a_true),
            np.array(s_logit), np.array(s_true))

val_v_logit, val_v_true, val_a_logit, val_a_true, val_s_logit, val_s_true = \
    collect_deterministic(eval_model, val_loader, need_sync=CONFIG["USE_CROSS_ATTENTION"])

def fit_temperature(logits, labels, max_iter=100, lr=0.05):
    if len(set(labels.astype(int).tolist())) < 2:
        return 1.0
    logits_t = torch.tensor(logits, dtype=torch.float32)
    labels_t = torch.tensor(labels, dtype=torch.float32)
    T = torch.nn.Parameter(torch.ones(1))
    opt = torch.optim.LBFGS([T], lr=lr, max_iter=max_iter)
    bce = nn.BCEWithLogitsLoss()
    def closure():
        opt.zero_grad()
        loss = bce(logits_t / T.clamp(min=1e-2), labels_t)
        loss.backward()
        return loss
    opt.step(closure)
    return float(T.clamp(min=1e-2).item())

temp_video = fit_temperature(val_v_logit, val_v_true)
temp_audio = fit_temperature(val_a_logit, val_a_true)
temp_sync = fit_temperature(val_s_logit, val_s_true) if len(val_s_true) else 1.0
print(f"Fitted temperatures -> video: {temp_video:.3f} | audio: {temp_audio:.3f} | sync: {temp_sync:.3f}")

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def youden_threshold(y_true, y_prob, default=0.5):
    """Single-draw Youden's-J threshold. Kept for Cell 6's DFDC-oracle
    threshold (an explicitly-labeled upper bound computed directly on the
    eval set itself, not transferred from elsewhere) where single-draw
    instability isn't the concern it is for a threshold that has to
    TRANSFER from val to a different split."""
    if len(set(int(v) for v in y_true)) < 2:
        return default
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    return float(thr[np.argmax(tpr - fpr)])

def bootstrap_stable_threshold(y_true, y_prob, n_boot=200, min_recall_floor=None, seed=42, default=0.5):
    """v3.3.1 fix: replaces a single Youden's-J point estimate with the
    MEDIAN threshold across n_boot bootstrap resamples of (y_true, y_prob).
    Motivation: the first v3.3 run's audio threshold (Youden's-J on 991 val
    samples) picked 0.1134, which gave real-audio recall of 0.06 on test --
    the reliability diagram showed test's real-audio scores are NOT
    clustered near 0, so a threshold sitting right at the edge of that
    range is highly sensitive to exactly which val samples happened to be
    drawn. A single point estimate on a modest val set is a high-variance
    estimator for a skewed/heavy-tailed score distribution; the median
    across resamples is far more stable, and the full bootstrap array lets
    us report that instability explicitly instead of hiding it.

    If min_recall_floor is given, any candidate cutoff whose implied
    recall on the class-0 (real) samples of THAT resample would fall below
    the floor is excluded before Youden's-J picks among what's left --  a
    hard backstop against ever repeating the 0.06-recall failure,
    independent of what pure Youden's-J would otherwise select.

    Returns (median_threshold, all_bootstrap_thresholds) -- the second
    value is for reporting stability, not for direct use.
    """
    y_true, y_prob = np.asarray(y_true), np.asarray(y_prob)
    if len(set(y_true.astype(int).tolist())) < 2:
        return default, np.array([])
    rng = np.random.RandomState(seed)
    n = len(y_true)
    thresholds = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yt, yp = y_true[idx], y_prob[idx]
        if len(set(yt.astype(int).tolist())) < 2:
            continue
        fpr, tpr, thr = roc_curve(yt, yp)
        j = tpr - fpr
        if min_recall_floor is not None:
            real_scores = yp[yt == 0]
            if len(real_scores) > 0:
                recall0_at = np.array([(real_scores < c).mean() for c in thr])
                valid = recall0_at >= min_recall_floor
                if valid.any():
                    j = np.where(valid, j, -np.inf)
        thresholds.append(float(thr[np.argmax(j)]))
    if not thresholds:
        return default, np.array([])
    thresholds = np.array(thresholds)
    return float(np.median(thresholds)), thresholds

# Video showed no sign of the threshold-instability failure mode in the
# first run (balanced precision/recall at its Youden cutoff), so it keeps
# a plain bootstrap median with no floor. Audio's real-class recall
# collapsed to 0.06 at the naive Youden point -- a 0.5 floor is a direct,
# targeted backstop against that specific failure. Sync isn't trained
# above chance yet (see Cell 4's diagnosis print), so its threshold barely
# matters right now, but it's bootstrapped the same way for consistency.
thr_video, thr_video_boot = bootstrap_stable_threshold(val_v_true, sigmoid(val_v_logit / temp_video))
thr_audio, thr_audio_boot = bootstrap_stable_threshold(
    val_a_true, sigmoid(val_a_logit / temp_audio), min_recall_floor=0.5)
thr_sync, thr_sync_boot = bootstrap_stable_threshold(val_s_true, sigmoid(val_s_logit / temp_sync)) \
    if len(val_s_true) else (0.5, np.array([]))

print(f"Bootstrap-median thresholds (fit on VAL, {len(thr_video_boot)} resamples) -> "
      f"video: {thr_video:.4f} | audio: {thr_audio:.4f} (min-recall floor 0.5) | sync: {thr_sync:.4f}")
if len(thr_video_boot):
    print(f"Threshold stability (bootstrap std across resamples, lower = more stable) -> "
          f"video: {thr_video_boot.std():.4f} | audio: {thr_audio_boot.std():.4f} | "
          f"sync: {thr_sync_boot.std() if len(thr_sync_boot) else float('nan'):.4f}")

# ---- Pass 2: MC-Dropout on VAL, to derive the uncertainty-flag cutoffs ----
def mc_predict_calibrated(model, visual, audio, mask, temps, n_samples=20, need_sync=False):
    model.eval()
    enable_mc_dropout(model)
    v_logits, a_logits, s_logits = [], [], []
    with torch.no_grad():
        for _ in range(n_samples):
            out = model(visual, audio, visual_mask=mask)
            v_logits.append(out["video_logit"].cpu().numpy())
            a_logits.append(out["audio_logit"].cpu().numpy())
            if need_sync and out["sync_logit"] is not None:
                s_logits.append(out["sync_logit"].cpu().numpy())
    model.eval()
    v_arr, a_arr = np.stack(v_logits, 0), np.stack(a_logits, 0)
    result = {
        "video": (sigmoid(v_arr.mean(0) / temps["video"]), sigmoid(v_arr).std(0)),
        "audio": (sigmoid(a_arr.mean(0) / temps["audio"]), sigmoid(a_arr).std(0)),
    }
    if s_logits:
        s_arr = np.stack(s_logits, 0)
        result["sync"] = (sigmoid(s_arr.mean(0) / temps["sync"]), sigmoid(s_arr).std(0))
    return result

temps = {"video": temp_video, "audio": temp_audio, "sync": temp_sync}
val_v_unc, val_a_unc = [], []
with torch.no_grad():
    for vis, audio, vlab, alab, mask in val_loader:
        vis, audio, mask = vis.to(device), audio.to(device), mask.to(device)
        res = mc_predict_calibrated(eval_model, vis, audio, mask, temps, n_samples=CONFIG["N_MC_SAMPLES"])
        val_v_unc.extend(res["video"][1]); val_a_unc.extend(res["audio"][1])

unc_flag_video = float(np.percentile(val_v_unc, 95))
unc_flag_audio = float(np.percentile(val_a_unc, 95))
print(f"Uncertainty-flag cutoffs (95th pct of VAL MC-Dropout std) -> video: {unc_flag_video:.4f} | "
      f"audio: {unc_flag_audio:.4f}")

calibration = {
    "thr_video": thr_video, "thr_audio": thr_audio, "thr_sync": thr_sync,
    "temp_video": temp_video, "temp_audio": temp_audio, "temp_sync": temp_sync,
    "unc_flag_video": unc_flag_video, "unc_flag_audio": unc_flag_audio,
}
with open("/kaggle/working/calibration.json", "w") as f:
    json.dump(calibration, f, indent=2)
# legacy-compatible single-value threshold files, for any older tooling
np.save("/kaggle/working/optimal_threshold_video.npy", np.array([thr_video]))
np.save("/kaggle/working/optimal_threshold_audio.npy", np.array([thr_audio]))
np.save("/kaggle/working/optimal_threshold.npy", np.array([thr_video]))
print("calibration.json written.")

# ---- Pass 3: FULL REPORT on the TEST split (touched here for the first/only time) ----
test_v_true, test_v_prob, test_v_unc = [], [], []
test_a_true, test_a_prob, test_a_unc = [], [], []
test_s_true, test_s_prob = [], []
with torch.no_grad():
    for vis, audio, vlab, alab, mask in test_loader:
        vis, audio, mask = vis.to(device), audio.to(device), mask.to(device)
        res = mc_predict_calibrated(eval_model, vis, audio, mask, temps,
                                     n_samples=CONFIG["N_MC_SAMPLES"], need_sync=CONFIG["USE_CROSS_ATTENTION"])
        mean_v, std_v = res["video"]; mean_a, std_a = res["audio"]
        test_v_prob.extend(mean_v); test_v_unc.extend(std_v); test_v_true.extend(vlab.numpy())
        test_a_prob.extend(mean_a); test_a_unc.extend(std_a); test_a_true.extend(alab.numpy())
        if "sync" in res:
            B = audio.size(0)
            perm = torch.roll(torch.arange(B, device=device), shifts=1)
            res_shuf = mc_predict_calibrated(eval_model, vis, audio[perm], mask, temps,
                                              n_samples=CONFIG["N_MC_SAMPLES"], need_sync=True)
            test_s_prob.extend(res["sync"][0]); test_s_prob.extend(res_shuf["sync"][0])
            test_s_true.extend([0] * B + [1] * B)

test_v_true, test_v_prob, test_v_unc = np.array(test_v_true, dtype=int), np.array(test_v_prob), np.array(test_v_unc)
test_a_true, test_a_prob, test_a_unc = np.array(test_a_true, dtype=int), np.array(test_a_prob), np.array(test_a_unc)
test_s_true, test_s_prob = np.array(test_s_true, dtype=int), np.array(test_s_prob)

v_pred = (test_v_prob >= thr_video).astype(int)
a_pred = (test_a_prob >= thr_audio).astype(int)

def expected_calibration_error(probs, labels, n_bins=15):
    """Standard binary ECE: gap between mean predicted confidence and the
    empirical FRACTION OF POSITIVES in each bin (retained from the
    v3.2.1 fix -- comparing against thresholded-prediction accuracy would
    again make a confidently-correct low-probability bin look badly
    miscalibrated)."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece, bin_fracs, bin_confs = 0.0, [], []
    for i in range(n_bins):
        lo = bins[i] if i > 0 else -0.01
        mask = (probs > lo) & (probs <= bins[i + 1])
        if mask.sum() > 0:
            bin_frac = labels[mask].mean()
            bin_conf = probs[mask].mean()
            ece += (mask.sum() / len(probs)) * abs(bin_frac - bin_conf)
            bin_fracs.append(bin_frac); bin_confs.append(bin_conf)
        else:
            bin_fracs.append(np.nan); bin_confs.append(np.nan)
    return ece, bin_fracs, bin_confs

def bootstrap_ci(y_true, y_score, metric_fn, n_boot=2000, seed=42):
    """Percentile bootstrap 95% CI for an arbitrary metric_fn(y_true, y_score)."""
    rng = np.random.RandomState(seed)
    y_true, y_score = np.asarray(y_true), np.asarray(y_score)
    n = len(y_true)
    stats = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yt, ys = y_true[idx], y_score[idx]
        if len(set(yt.tolist())) < 2:
            continue
        try:
            stats.append(metric_fn(yt, ys))
        except ValueError:
            continue
    if not stats:
        return float('nan'), float('nan'), float('nan')
    return float(np.mean(stats)), float(np.percentile(stats, 2.5)), float(np.percentile(stats, 97.5))

print("\n" + "=" * 65)
print("  CROSSFUSE v3.3 TEST-SET REPORT (calibrated, thresholds fit on VAL)")
print(f"  Video thr: {thr_video:.4f} (T={temp_video:.2f}) | Audio thr: {thr_audio:.4f} (T={temp_audio:.2f}) | "
      f"Sync thr: {thr_sync:.4f} (T={temp_sync:.2f})")
print("=" * 65)

print("\n--- VIDEO HEAD (is the visual track fake?) ---")
print(classification_report(test_v_true, v_pred, target_names=["Real vid", "Fake vid"], zero_division=0))
v_auc_mean, v_auc_lo, v_auc_hi = bootstrap_ci(test_v_true, test_v_prob, roc_auc_score)
print(f"Video AUC (test, bootstrap 95% CI): {v_auc_mean:.4f} [{v_auc_lo:.4f}, {v_auc_hi:.4f}]")
print(f"Mean MC-Dropout uncertainty (video): {test_v_unc.mean():.4f}")

print("\n--- AUDIO HEAD (is the audio track fake?) ---")
print(classification_report(test_a_true, a_pred, target_names=["Real aud", "Fake aud"], zero_division=0))
a_auc_mean, a_auc_lo, a_auc_hi = bootstrap_ci(test_a_true, test_a_prob, roc_auc_score)
print(f"Audio AUC (test, bootstrap 95% CI): {a_auc_mean:.4f} [{a_auc_lo:.4f}, {a_auc_hi:.4f}]")
print(f"Mean MC-Dropout uncertainty (audio): {test_a_unc.mean():.4f}")

if len(test_s_true):
    s_pred = (test_s_prob >= thr_sync).astype(int)
    print("\n--- SYNC HEAD (are audio and video mismatched?) [held-out shuffled-pair eval] ---")
    print(classification_report(test_s_true, s_pred, target_names=["Matched", "Mismatched"], zero_division=0))
    s_auc_mean, s_auc_lo, s_auc_hi = bootstrap_ci(test_s_true, test_s_prob, roc_auc_score)
    print(f"Sync AUC (test, bootstrap 95% CI): {s_auc_mean:.4f} [{s_auc_lo:.4f}, {s_auc_hi:.4f}]  "
          f"(near 0.5 would mean cross-attention is still decorative)")

ece_v, bacc_v, bconf_v = expected_calibration_error(test_v_prob, test_v_true)
ece_a, bacc_a, bconf_a = expected_calibration_error(test_a_prob, test_a_true)
print(f"\nExpected Calibration Error (test) -> Video: {ece_v:.4f} | Audio: {ece_a:.4f}  (lower is better)")

# ---- Per-head plots (test set): confusion / ROC / reliability ----
sns.set_theme(style="whitegrid", font_scale=1.0)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for row, (name, yt, yp, pred, thr, ece, bacc, bconf) in enumerate([
        ("Video", test_v_true, test_v_prob, v_pred, thr_video, ece_v, bacc_v, bconf_v),
        ("Audio", test_a_true, test_a_prob, a_pred, thr_audio, ece_a, bacc_a, bconf_a)]):
    cm = confusion_matrix(yt, pred, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[row, 0],
                xticklabels=["Pred real", "Pred fake"], yticklabels=["Actual real", "Actual fake"],
                annot_kws={"size": 14})
    axes[row, 0].set_title(f"{name} Head -- Confusion (test, p>={thr:.2f})", fontsize=12, fontweight='bold')

    if len(set(int(v) for v in yt)) > 1:
        fpr, tpr, thrs = roc_curve(yt, yp)
        roc_auc = auc(fpr, tpr)
        oi = np.argmax(tpr - fpr)
        axes[row, 1].plot(fpr, tpr, color='#1f77b4', lw=3, label=f'{name} (AUC={roc_auc:.3f})')
        axes[row, 1].scatter(fpr[oi], tpr[oi], color='red', s=90, zorder=5, label=f'Cutoff {thr:.2f}')
    axes[row, 1].plot([0, 1], [0, 1], color='gray', lw=1.2, linestyle='--')
    axes[row, 1].set_xlabel('FPR'); axes[row, 1].set_ylabel('TPR')
    axes[row, 1].set_title(f'{name} Head -- ROC (test)', fontsize=12, fontweight='bold')
    axes[row, 1].legend(loc="lower right")

    axes[row, 2].plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    axes[row, 2].plot(bconf, bacc, marker='o', color='#d62728', label=f'{name} (ECE={ece:.3f})')
    axes[row, 2].set_xlabel('Mean predicted confidence')
    axes[row, 2].set_ylabel('Empirical fraction of positives')   # v3.3 fix: was mislabeled "accuracy"
    axes[row, 2].set_title(f'{name} Head -- Reliability (test)', fontsize=12, fontweight='bold')
    axes[row, 2].legend(loc="upper left")

plt.tight_layout()
plt.savefig("/kaggle/working/crossfuse_v33_perhead_plots.png", dpi=300, bbox_inches='tight')
plt.show()

# ---- Sync-head ROC (evidence cross-attention is load-bearing) ----
if len(test_s_true):
    fpr_s, tpr_s, _ = roc_curve(test_s_true, test_s_prob)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr_s, tpr_s, color='#2ca02c', lw=3, label=f'Sync (AUC={auc(fpr_s, tpr_s):.3f})')
    plt.plot([0, 1], [0, 1], color='gray', lw=1.2, linestyle='--')
    plt.xlabel('FPR'); plt.ylabel('TPR')
    plt.title('AV Sync Head -- ROC (test, shuffled-pair eval)', fontsize=13, fontweight='bold')
    plt.legend(loc="lower right"); plt.tight_layout()
    plt.savefig("/kaggle/working/crossfuse_v33_sync_roc.png", dpi=300, bbox_inches='tight')
    plt.show()

# ---- 4-way attribution confusion (derived category from the two heads, test set) ----
def to_category_idx(vf, af):
    return {(0, 0): 0, (0, 1): 1, (1, 0): 2, (1, 1): 3}[(int(vf), int(af))]

cat_names = ["RVRA\n(real/real)", "RVFA\n(real/fakeAud)", "FVRA\n(fakeVid/real)", "FVFA\n(fake/fake)"]
true_cat = [to_category_idx(vt, at) for vt, at in zip(test_v_true, test_a_true)]
pred_cat = [to_category_idx(vp, ap) for vp, ap in zip(v_pred, a_pred)]
cm4 = confusion_matrix(true_cat, pred_cat, labels=[0, 1, 2, 3])

plt.figure(figsize=(7.5, 6))
sns.heatmap(cm4, annot=True, fmt='d', cmap='Greens', cbar=False,
            xticklabels=cat_names, yticklabels=cat_names, annot_kws={"size": 12})
plt.title("Modality Attribution -- Derived 4-Category Confusion (test)", fontsize=13, fontweight='bold', pad=12)
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.tight_layout()
plt.savefig("/kaggle/working/crossfuse_v33_attribution_matrix.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"\nThresholds (val) -> video: {thr_video:.4f}, audio: {thr_audio:.4f}, sync: {thr_sync:.4f}")
print("All classification reports, ECE, ROC/reliability plots and the attribution matrix above are "
      "computed on the TEST split, which was not used for checkpoint selection or calibration.")

### Cell 6 — DFDC Zero-Shot Benchmark, 300/300 + Bootstrap CI
Same primary framing as v3.2 (video head vs. the DFDC video label, generalization gap = val AUC − DFDC
AUC) but on 300/300 clips instead of 100/100 (narrows the 95% CI from ±0.09 to roughly ±0.045), through
the same batched/masked/ffmpeg pipeline as Cell 3, reporting both the transferred-threshold accuracy and
a DFDC-oracle upper bound. The v3.2 stale-tensor-cache purge (a real bug it caught) is retained.

In [ ]:
# =====================================================================
# CELL 6: DFDC ZERO-SHOT BENCHMARK, 300 REAL / 300 FAKE + BOOTSTRAP CI (v3.3)
# DFDC is a VIDEO benchmark (per-clip real/fake, audio usually genuine),
# so the primary zero-shot number is the calibrated VIDEO head evaluated
# against the DFDC label.
#
# WHAT CHANGED vs v3.2:
#   * 300/300 instead of 100/100 -- the v3.2 DFDC headline (0.627 AUC)
#     rested on 154 clips, a 95% CI of roughly +/-0.09 wide enough that
#     it wasn't a defensible point estimate on its own. 600 clips
#     narrows that to roughly +/-0.045, and we now report the CI
#     explicitly via the same bootstrap_ci() used for the test-set report.
#   * Uses the SAME ffmpeg_extract_wav() + batched, 2-GPU
#     extract_visual_embeddings_batched() pipeline as Cell 3, instead of
#     a separately hand-rolled per-frame loop -- one fewer place for the
#     video/audio pipelines to silently drift apart.
#   * Reports the video head at BOTH the threshold transferred from val
#     (the honest zero-shot number) AND the DFDC-oracle threshold (an
#     upper bound showing how much of the gap is calibration vs.
#     discrimination).
#   * Reports post-temperature-scaling ECE (using temp_video from Cell 5)
#     instead of the uncalibrated probability.
#   * make_rng("dfdc") for the sample draw -- reproducible independent of
#     how many other random calls happened earlier in the run (v3.2 used
#     the shared global `random` stream).
#   * The v3.2 stale-cache purge (a real bug catch -- a prior run had left
#     54 old tensors that silently leaked into the eval set) is retained.
# =====================================================================

import json
import os
import glob
import numpy as np
import torch
from tqdm import tqdm
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

dfdc_output_dir = "/kaggle/working/dfdc_test_features_v33"
os.makedirs(dfdc_output_dir, exist_ok=True)

# build_extractor(), extract_visual_embeddings_batched(), extract_audio_mfcc(),
# img_transform, N_DEVICES and DEVICE_IDS all come from Cell 1, so they're
# in scope whether or not Cell 3 (Stage A) ran in this session. Only the
# `extractors` dict itself (one MTCNN+EfficientNet-B4 pair per GPU) needs
# building here if Cell 3 didn't already build it.
if 'extractors' not in globals():
    print("[i] Building fresh extractors for DFDC (Cell 3 didn't run in this session).")
    extractors = {d: build_extractor(d) for d in DEVICE_IDS}

def load_dfdc_ground_truth(max_per_class=300, rng=None):
    """(path, label) tuples travel together through every shuffle/slice."""
    labeled = []
    meta_files = sorted(glob.glob("/kaggle/input/**/metadata.json", recursive=True))
    for meta_path in meta_files:
        folder = os.path.dirname(meta_path)
        with open(meta_path, 'r') as f:
            data = json.load(f)
        for vid_name, info in data.items():
            full_path = os.path.join(folder, vid_name)
            if not os.path.exists(full_path):
                continue
            lbl = info.get('label', '').upper()
            if lbl == 'REAL':
                labeled.append((full_path, 0))
            elif lbl == 'FAKE':
                labeled.append((full_path, 1))

    reals = [x for x in labeled if x[1] == 0]
    fakes = [x for x in labeled if x[1] == 1]
    rng.shuffle(reals)
    rng.shuffle(fakes)
    limit = min(max_per_class, len(reals), len(fakes))
    if limit < max_per_class:
        print(f"[!] Only {limit} clips/class available (requested {max_per_class}) -- "
              f"CI will be wider than the 300/300 target.")
    selected = reals[:limit] + fakes[:limit]
    rng.shuffle(selected)
    return selected

rng_dfdc = make_rng("dfdc")
selected_dfdc = load_dfdc_ground_truth(max_per_class=300, rng=rng_dfdc)
print(f"Extracting features for {len(selected_dfdc)} unseen DFDC videos "
      f"({sum(1 for _, l in selected_dfdc if l == 0)} real / {sum(1 for _, l in selected_dfdc if l == 1)} fake)...")

# Purge stale cached tensors from any earlier run's DIFFERENT sample --
# real bug caught in v3.2 (54 stale tensors silently inflated a prior
# eval set from 77/77 to 77/131).
expected_names = {"dfdc_" + os.path.basename(p).split('.')[0] + ".pt" for p, _ in selected_dfdc}
existing_names = set(os.listdir(dfdc_output_dir)) if os.path.isdir(dfdc_output_dir) else set()
stale_names = existing_names - expected_names
if stale_names:
    print(f"Removing {len(stale_names)} stale cached tensor(s) left over from a previous run...")
    for fname in stale_names:
        os.remove(os.path.join(dfdc_output_dir, fname))

def process_dfdc_job(video_path, label, device_str, mtcnn_local, eff_local):
    video_name = "dfdc_" + os.path.basename(video_path).split('.')[0]
    save_path = os.path.join(dfdc_output_dir, f"{video_name}.pt")
    if os.path.exists(save_path):
        return "cached"
    visual, n_valid = extract_visual_embeddings_batched(
        video_path, mtcnn_local, eff_local, device_str,
        apply_self_blend=False, augment_fn=None, rng=None, n_frames=CONFIG["VISUAL_FRAMES"])
    if visual is None:
        return "no_faces"
    if visual.shape[0] < CONFIG["VISUAL_FRAMES"]:
        pad = torch.zeros((CONFIG["VISUAL_FRAMES"] - visual.shape[0], visual.shape[1]), dtype=visual.dtype)
        visual = torch.cat([visual, pad], dim=0)
    else:
        visual = visual[:CONFIG["VISUAL_FRAMES"]]
    audio = extract_audio_mfcc(video_path)
    # DFDC only has a per-clip VIDEO label -> stored as 'label' and
    # 'video_label'; 'audio_label'=0 (DFDC audio is generally genuine).
    torch.save({'visual': visual, 'audio': torch.tensor(audio, dtype=torch.float32),
                'label': label, 'video_label': label, 'audio_label': 0,
                'n_valid_frames': n_valid}, save_path)
    return "ok"

with ThreadPoolExecutor(max_workers=len(DEVICE_IDS)) as pool:
    futures = []
    for i, (video_path, label) in enumerate(selected_dfdc):
        d = DEVICE_IDS[i % len(DEVICE_IDS)]
        mtcnn_local, eff_local = extractors[d]
        futures.append(pool.submit(process_dfdc_job, video_path, label, d, mtcnn_local, eff_local))
    dfdc_status = defaultdict(int)
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Extracting DFDC (v3.3, 2-GPU batched)"):
        dfdc_status[fut.result()] += 1
print(f"DFDC extraction outcomes: {dict(dfdc_status)}")

dfdc_dataset = CrossFuseDataset(dfdc_output_dir, manifest_path=None, target_frames=CONFIG["VISUAL_FRAMES"],
                                 use_cmvn=CONFIG["USE_MFCC_CMVN"])  # CMVN applied at load time, see Cell 1
dfdc_loader = DataLoader(dfdc_dataset, batch_size=16, shuffle=False)

eval_model.eval()  # reuse the calibrated model + temps/thresholds from Cell 5

dfdc_labels, dfdc_vprob, dfdc_vunc = [], [], []
with torch.no_grad():
    for vis_batch, audio_batch, vlab_batch, alab_batch, mask_batch in dfdc_loader:
        vis_batch, audio_batch, mask_batch = vis_batch.to(device), audio_batch.to(device), mask_batch.to(device)
        res = mc_predict_calibrated(eval_model, vis_batch, audio_batch, mask_batch, temps,
                                     n_samples=CONFIG["N_MC_SAMPLES"])
        dfdc_labels.extend(vlab_batch.numpy())
        dfdc_vprob.extend(res["video"][0])
        dfdc_vunc.extend(res["video"][1])

dfdc_labels = np.array(dfdc_labels, dtype=int)
dfdc_vprob = np.array(dfdc_vprob)

# Primary: video head at the threshold TRANSFERRED from val (the honest zero-shot number)
video_preds = (dfdc_vprob >= thr_video).astype(int)
# Secondary: DFDC-oracle threshold -- upper bound isolating discrimination from calibration
thr_video_oracle = youden_threshold(dfdc_labels, dfdc_vprob, default=thr_video)
video_preds_oracle = (dfdc_vprob >= thr_video_oracle).astype(int)

print("\n" + "=" * 60)
print(f"  UNSEEN DFDC REPORT -- v3.3 (VIDEO head, transferred threshold {thr_video:.2f})")
print("=" * 60)
print(classification_report(dfdc_labels, video_preds, target_names=["Real (0)", "Fake (1)"], zero_division=0))

dfdc_video_acc = accuracy_score(dfdc_labels, video_preds)
dfdc_auc_mean, dfdc_auc_lo, dfdc_auc_hi = bootstrap_ci(dfdc_labels, dfdc_vprob, roc_auc_score)
dfdc_ece, _, _ = expected_calibration_error(dfdc_vprob, dfdc_labels)
oracle_acc = accuracy_score(dfdc_labels, video_preds_oracle)

print(f">> Video-head Zero-Shot Accuracy on DFDC (transferred thr) : {dfdc_video_acc*100:.2f}%")
print(f">> Video-head Zero-Shot AUC-ROC on DFDC (bootstrap 95% CI)  : "
      f"{dfdc_auc_mean:.4f} [{dfdc_auc_lo:.4f}, {dfdc_auc_hi:.4f}]")
print(f">> Video-head Zero-Shot ECE on DFDC (post-temperature)      : {dfdc_ece:.4f}")
print(f">> Mean MC-Dropout uncertainty (video)                      : {np.mean(dfdc_vunc):.4f}")
print(f">> [upper bound] Accuracy at DFDC-oracle threshold {thr_video_oracle:.2f}      : {oracle_acc*100:.2f}%")

with open("/kaggle/working/best_metrics.json") as f:
    best_metrics = json.load(f)
generalization_gap = best_metrics["best_val_video_auc"] - dfdc_auc_mean
print(f"\n>> CROSS-DATASET GENERALIZATION GAP (Val Video AUC {best_metrics['best_val_video_auc']:.4f} - "
      f"DFDC Video AUC {dfdc_auc_mean:.4f}): {generalization_gap:.4f}")
print("   Headline Table 3 number -- compare against the ablation-runner (Cell 8) row A6 "
      "(USE_COMPRESSION_AUG=False) to see what the compression-augmented replicas bought.")

plt.figure(figsize=(6, 5))
cm = confusion_matrix(dfdc_labels, video_preds, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', cbar=False,
            xticklabels=["Pred Real", "Pred Fake"], yticklabels=["Actual Real", "Actual Fake"], annot_kws={"size": 14})
plt.title(f"DFDC Generalization (Video head, AUC = {dfdc_auc_mean:.4f} "
          f"[{dfdc_auc_lo:.3f}, {dfdc_auc_hi:.3f}])", fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig("/kaggle/working/dfdc_generalization_matrix_v33.png", dpi=300)
plt.show()

dfdc_results = {
    "n_real": int((dfdc_labels == 0).sum()), "n_fake": int((dfdc_labels == 1).sum()),
    "video_auc_mean": dfdc_auc_mean, "video_auc_ci": [dfdc_auc_lo, dfdc_auc_hi],
    "video_acc_transferred_thr": dfdc_video_acc, "video_acc_oracle_thr": oracle_acc,
    "video_ece": dfdc_ece, "generalization_gap": generalization_gap,
}
with open("/kaggle/working/dfdc_results_v33.json", "w") as f:
    json.dump(dfdc_results, f, indent=2)
print("\ndfdc_results_v33.json written.")

### Cell 7 — Interactive Inference: Per-Modality + AV-Sync Verdict
Same upload-and-analyze workflow, now reporting a third signal — `P(AV mismatch)` from the sync head —
alongside `P(video fake)` and `P(audio fake)`. Audio goes through the same `ffmpeg_extract_wav()` front
end as training (the direct fix for the earlier false-positive audio verdict), and all thresholds /
temperatures / uncertainty cutoffs load from `calibration.json` rather than being hardcoded.

In [ ]:
# =====================================================================
# CELL 7: INTERACTIVE IN-THE-WILD INFERENCE -- PER-MODALITY + AV-SYNC
#         VERDICT (v3.3)
#
# WHAT CHANGED vs v3.2:
#   * Audio now goes through the SAME ffmpeg_extract_wav() front-end as
#     Cells 3 and 6. This is the direct fix for the earlier in-the-wild
#     run's "PySoundFile failed. Trying audioread instead" warning and
#     the resulting 71.8% audio-fake false positive -- that clip's audio
#     was decoded through a different backend than every training
#     tensor; it no longer can be.
#   * Thresholds, temperatures AND per-head uncertainty-flag cutoffs are
#     all loaded from calibration.json (Cell 5) instead of a hardcoded
#     UNCERTAINTY_FLAG_THRESHOLD=0.15 -- v3.2's val uncertainty
#     distribution was actually 0.035 (video) / 0.082 (audio), so 0.15
#     was firing the "flagged for review" branch fairly arbitrarily.
#   * Reports a third signal, P(AV mismatch) from the sync head, which
#     escalates an otherwise-clean clip to "AV streams inconsistent" --
#     this is the payoff of making cross-attention load-bearing (Cell 2).
#   * MTCNN/EfficientNet-B4/model are now cached at module scope instead
#     of being rebuilt from scratch on every single upload.
#   * Fixed a real NameError: v3.2's on_file_upload left `content`
#     undefined if upload_widget.value was neither a tuple/list nor a
#     dict (which is what ipywidgets' newer FileUpload trait actually
#     returns in some versions) -- added an explicit else + validation.
# =====================================================================

import os
import io
import json
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
from torchvision import transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from facenet_pytorch import MTCNN

with open("/kaggle/working/calibration.json") as f:
    _calib = json.load(f)
THR_VIDEO, THR_AUDIO, THR_SYNC = _calib["thr_video"], _calib["thr_audio"], _calib["thr_sync"]
TEMP_VIDEO, TEMP_AUDIO, TEMP_SYNC = _calib["temp_video"], _calib["temp_audio"], _calib["temp_sync"]
UNC_FLAG_VIDEO, UNC_FLAG_AUDIO = _calib["unc_flag_video"], _calib["unc_flag_audio"]
INFER_TEMPS = {"video": TEMP_VIDEO, "audio": TEMP_AUDIO, "sync": TEMP_SYNC}

print(f"Loaded calibration -> thr(v/a/s)={THR_VIDEO:.3f}/{THR_AUDIO:.3f}/{THR_SYNC:.3f} | "
      f"unc_flag(v/a)={UNC_FLAG_VIDEO:.4f}/{UNC_FLAG_AUDIO:.4f}")

# ---- Module-scope cached backbone + model (built once, reused per upload) ----
_infer_mtcnn = MTCNN(image_size=224, margin=20, keep_all=False, device=device)
_infer_eff = efficientnet_b4(weights=EfficientNet_B4_Weights.DEFAULT)
_infer_eff.classifier = nn.Identity()
_infer_eff = _infer_eff.to(device).eval()
_infer_img_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
_infer_model = CrossFuseModel(visual_dim=1792, target_seq_len=CONFIG["VISUAL_FRAMES"],
                               use_cross_attention=CONFIG["USE_CROSS_ATTENTION"]).to(device)
_infer_model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=False))
_infer_model.eval()


def analyze_custom_video(video_path: str):
    print("\n" + "=" * 65)
    print("      CROSSFUSE v3.3 IN-THE-WILD FORENSIC INFERENCE ENGINE")
    print("=" * 65)

    if not os.path.exists(video_path):
        raise FileNotFoundError(f"ERROR: Could not find video at '{video_path}'.")

    print(f"[*] Target Video      : {os.path.basename(video_path)}")
    print(f"[*] Hardware Device   : {device.type.upper()}")

    print("[*] Sampling facial embeddings (batched)...")
    vis_tensor, n_valid = extract_visual_embeddings_batched(
        video_path, _infer_mtcnn, _infer_eff, str(device),
        apply_self_blend=False, augment_fn=None, rng=None, n_frames=CONFIG["VISUAL_FRAMES"])
    if vis_tensor is None:
        raise RuntimeError("ERROR: Could not extract any valid face crops from the video!")
    if vis_tensor.shape[0] < CONFIG["VISUAL_FRAMES"]:
        pad = torch.zeros((CONFIG["VISUAL_FRAMES"] - vis_tensor.shape[0], vis_tensor.shape[1]), dtype=vis_tensor.dtype)
        vis_tensor = torch.cat([vis_tensor, pad], dim=0)
    else:
        vis_tensor = vis_tensor[:CONFIG["VISUAL_FRAMES"]]
    visual_mask = torch.zeros(1, CONFIG["VISUAL_FRAMES"], dtype=torch.bool)
    visual_mask[0, :max(1, n_valid)] = True

    print("[*] Extracting acoustic speech MFCCs (ffmpeg front-end, full-clip, time-aligned)...")
    raw_mfcc = extract_audio_mfcc(video_path)                     # raw (Cell 3 no longer bakes in CMVN)
    aud_tensor = torch.tensor(maybe_cmvn(raw_mfcc, CONFIG["USE_MFCC_CMVN"]), dtype=torch.float32)

    print("[*] Running two-head + AV-sync inference with MC-Dropout uncertainty...")
    vis_input = vis_tensor.unsqueeze(0).to(device)
    aud_input = aud_tensor.unsqueeze(0).to(device)
    mask_input = visual_mask.to(device)

    res = mc_predict_calibrated(_infer_model, vis_input, aud_input, mask_input, INFER_TEMPS,
                                 n_samples=CONFIG["N_MC_SAMPLES"], need_sync=CONFIG["USE_CROSS_ATTENTION"])
    p_video_fake, unc_video = float(res["video"][0][0]), float(res["video"][1][0])
    p_audio_fake, unc_audio = float(res["audio"][0][0]), float(res["audio"][1][0])
    p_av_mismatch = float(res["sync"][0][0]) if "sync" in res else None

    video_is_fake = p_video_fake >= THR_VIDEO
    audio_is_fake = p_audio_fake >= THR_AUDIO
    av_mismatched = (p_av_mismatch is not None) and (p_av_mismatch >= THR_SYNC)
    video_uncertain = unc_video >= UNC_FLAG_VIDEO
    audio_uncertain = unc_audio >= UNC_FLAG_AUDIO

    if video_uncertain or audio_uncertain:
        which = []
        if video_uncertain: which.append("video")
        if audio_uncertain: which.append("audio")
        verdict = f"UNCERTAIN ({'/'.join(which)}) -- flagged for human review"
    elif video_is_fake and audio_is_fake:
        verdict = "DEEPFAKE -- video + audio BOTH manipulated"
    elif video_is_fake:
        verdict = "DEEPFAKE -- VIDEO track manipulated (audio looks real)"
    elif audio_is_fake:
        verdict = "DEEPFAKE -- AUDIO track manipulated (video looks real)"
    elif av_mismatched:
        verdict = "SUSPICIOUS -- audio and video streams do not correspond (flagged for review)"
    else:
        verdict = "AUTHENTIC MEDIA (video and audio both real, streams correspond)"

    print("\n" + "-" * 65)
    print("                     FORENSIC ANALYSIS REPORT (v3.3)")
    print("-" * 65)
    print(f"  > P(video fake)     : {p_video_fake*100:6.2f}%   (thr {THR_VIDEO*100:.1f}%, uncertainty {unc_video:.4f}, flag {UNC_FLAG_VIDEO:.4f})")
    print(f"  > P(audio fake)     : {p_audio_fake*100:6.2f}%   (thr {THR_AUDIO*100:.1f}%, uncertainty {unc_audio:.4f}, flag {UNC_FLAG_AUDIO:.4f})")
    if p_av_mismatch is not None:
        print(f"  > P(AV mismatch)    : {p_av_mismatch*100:6.2f}%   (thr {THR_SYNC*100:.1f}%)")
    print("-" * 65)
    print(f"  > VIDEO verdict     : {'FAKE' if video_is_fake else 'REAL'}")
    print(f"  > AUDIO verdict     : {'FAKE' if audio_is_fake else 'REAL'}")
    print(f"  > FINAL VERDICT     : {verdict}")
    print("=" * 65 + "\n")


upload_widget = widgets.FileUpload(accept='.mp4,.avi,.mov', multiple=False, description='Upload Video', button_style='primary')
output_area = widgets.Output()

def on_file_upload(change):
    with output_area:
        clear_output()
        if not upload_widget.value:
            return
        print("Processing uploaded file...")
        uploaded_data = upload_widget.value
        file_name, content = None, None
        if isinstance(uploaded_data, (tuple, list)):
            file_info = uploaded_data[0]
            file_name = file_info['name']
            content = file_info['content']
        elif isinstance(uploaded_data, dict):
            file_name = list(uploaded_data.keys())[0]
            content = uploaded_data[file_name]['content']
        else:
            # v3.3 fix: v3.2 left `content` undefined here (NameError on
            # write), which happens with the FileUpload trait shape some
            # ipywidgets versions actually return. Fail loudly instead.
            print(f"[!] ERROR: unrecognized upload_widget.value type: {type(uploaded_data)}. "
                  f"Cannot extract file content.")
            return
        if file_name is None or content is None:
            print("[!] ERROR: could not parse uploaded file name/content.")
            return

        save_path = os.path.join("/kaggle/working", f"interactive_{file_name}")
        if isinstance(content, memoryview):
            content = content.tobytes()
        with open(save_path, "wb") as f:
            f.write(content)
        print(f"File saved cleanly to: {save_path}")
        analyze_custom_video(save_path)

upload_widget.observe(on_file_upload, names='value')
print("Click the button below to upload your test video directly:")
display(widgets.VBox([upload_widget, output_area]))

### Cell 8 (NEW) — Ablation Grid (A1–A8)
Reuses cached features and `run_training_pipeline()` to train 7 additional lightweight configurations
(A4 is the Cell 4 main run, reused rather than retrained) isolating the contribution of cross-attention,
hard negatives, compression augmentation, CMVN, and identity-disjoint vs. random splitting. **A3 vs A4**
is the row that justifies the paper's title; **A8 vs A4** quantifies the identity-leakage inflation
behind the old ~89% number.

In [ ]:
# =====================================================================
# CELL 8 (NEW in v3.3): ABLATION RUNNER (A1-A8)
# Each row reuses the SAME cached features (Cell 3) and the SAME
# run_training_pipeline() function (Cell 4) with a modified CONFIG copy
# -- only the loss composition / dataset filter / split strategy changes
# per row, so this cell trains 7 additional lightweight models (A4 IS
# the main run from Cell 4, reused rather than retrained) instead of
# re-running feature extraction.
#
#   A1  Video-only (no audio branch in the loss, no cross-attention)  -> unimodal baseline
#   A2  Audio-only                                                    -> unimodal baseline
#   A3  Two heads, NO cross-attention, OR-rule late fusion            -> the fusion baseline
#   A4  FULL CrossFuse (two heads + CA sync head)                     -> headline (= Cell 4's main run)
#   A5  A4 minus self-blend hard negatives                            -> value of hard negatives
#   A6  A4 minus compression augmentation                             -> value of the DFDC-gap fix
#   A7  A4 minus MFCC CMVN                                            -> value of audio normalization
#   A8  A4 with a RANDOM (non-identity-disjoint) split                -> quantifies the old "89%" illusion
#
# A3 vs A4 is the row that justifies the paper's "Cross-Attention
# Fusion" title. A8 lets you write the honest sentence: "a random split
# inflates video AUC from X to Y; we attribute the difference to
# identity leakage" -- the exact effect that produced v1/v2's ~89%.
#
# Metrics here are QUICK test-set AUCs from a single deterministic
# forward pass (no MC-Dropout averaging, no bootstrap CI, no temperature
# scaling) -- adequate for ranking configurations against each other,
# NOT a substitute for Cell 5/6's calibrated, bootstrapped headline
# numbers on the main run.
# =====================================================================

import copy
import csv
import json
import numpy as np
import torch
from sklearn.metrics import accuracy_score

def quick_test_eval(result):
    model_local = result["model"]
    model_local.eval()
    v_true, v_prob, a_true, a_prob = [], [], [], []
    with torch.no_grad():
        for vis, audio, vlab, alab, mask in result["test_loader"]:
            vis, audio, mask = vis.to(device), audio.to(device), mask.to(device)
            out = model_local(vis, audio, visual_mask=mask)
            v_prob.extend(torch.sigmoid(out["video_logit"]).cpu().numpy())
            a_prob.extend(torch.sigmoid(out["audio_logit"]).cpu().numpy())
            v_true.extend(vlab.numpy())
            a_true.extend(alab.numpy())
    v_true, v_prob = np.array(v_true), np.array(v_prob)
    a_true, a_prob = np.array(a_true), np.array(a_prob)
    video_auc = safe_auc(v_true, v_prob)
    audio_auc = safe_auc(a_true, a_prob)
    v_pred, a_pred = (v_prob >= 0.5).astype(int), (a_prob >= 0.5).astype(int)
    combined_pred = ((v_pred == 1) | (a_pred == 1)).astype(int)
    overall_true = ((v_true == 1) | (a_true == 1)).astype(int)
    combined_acc = accuracy_score(overall_true, combined_pred) if len(overall_true) else float('nan')
    return {"video_auc": video_auc, "audio_auc": audio_auc, "mean_auc": 0.5 * (video_auc + audio_auc),
            "combined_acc": combined_acc, "n_test": len(v_true)}


ablation_rows = []

def cfg_variant(**overrides):
    c = copy.deepcopy(CONFIG)
    c.update(overrides)
    return c

ABLATIONS = [
    ("A1_video_only", cfg_variant(MODALITY="video_only", USE_CROSS_ATTENTION=False, USE_SYNC_HEAD=False)),
    ("A2_audio_only", cfg_variant(MODALITY="audio_only", USE_CROSS_ATTENTION=False, USE_SYNC_HEAD=False)),
    ("A3_two_head_no_CA", cfg_variant(USE_CROSS_ATTENTION=False, USE_SYNC_HEAD=False)),
    # A4 is the Cell 4 main run -- reused below, not retrained.
    ("A5_no_hard_negatives", cfg_variant(GENERATE_HARD_NEGATIVES=False)),
    ("A6_no_compression_aug", cfg_variant(USE_COMPRESSION_AUG=False)),
    ("A7_no_mfcc_cmvn", cfg_variant(USE_MFCC_CMVN=False)),
    ("A8_random_split", cfg_variant(IDENTITY_DISJOINT_SPLIT=False)),
]

print(f"Running {len(ABLATIONS)} ablation configurations (A4 reuses the Cell 4 main run)...\n")

# A4 first, from the already-trained main run -- zero extra training cost.
a4_metrics = quick_test_eval(_main_result)
ablation_rows.append({"row": "A4_full_crossfuse", **a4_metrics,
                       "val_video_auc": _main_result["best_val_video_auc"],
                       "val_audio_auc": _main_result["best_val_audio_auc"],
                       "val_sync_auc": _main_result["best_val_sync_auc"]})
print(f"A4_full_crossfuse (= main run) -> video_auc={a4_metrics['video_auc']:.4f} | "
      f"audio_auc={a4_metrics['audio_auc']:.4f} | combined_acc={a4_metrics['combined_acc']:.4f}")

for tag, cfg in ABLATIONS:
    result = run_training_pipeline(cfg, tag=tag, verbose=True)
    metrics = quick_test_eval(result)
    ablation_rows.append({"row": tag, **metrics,
                           "val_video_auc": result["best_val_video_auc"],
                           "val_audio_auc": result["best_val_audio_auc"],
                           "val_sync_auc": result["best_val_sync_auc"]})
    print(f"{tag} -> video_auc={metrics['video_auc']:.4f} | audio_auc={metrics['audio_auc']:.4f} | "
          f"combined_acc={metrics['combined_acc']:.4f}\n")
    # Free GPU memory between ablation rows -- 8 models is a lot to keep resident.
    del result
    torch.cuda.empty_cache()

ablation_csv_path = "/kaggle/working/ablation_results.csv"
with open(ablation_csv_path, "w", newline="") as f:
    fieldnames = ["row", "video_auc", "audio_auc", "mean_auc", "combined_acc", "n_test",
                  "val_video_auc", "val_audio_auc", "val_sync_auc"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(ablation_rows)

print(f"\nAblation grid complete -> {ablation_csv_path}")
for row in ablation_rows:
    print(f"  {row['row']:<24} video_auc={row['video_auc']:.4f}  audio_auc={row['audio_auc']:.4f}  "
          f"combined_acc={row['combined_acc']:.4f}")

print("\nKey comparisons for the paper:")
print("  A3 (no cross-attention) vs A4 (full CrossFuse) -> justifies the 'Cross-Attention Fusion' title.")
print("  A8 (random split) vs A4 (identity-disjoint)     -> quantifies the identity-leakage inflation "
      "that produced the old ~89% number.")
print("  A6 (no compression aug) vs Cell 6's DFDC AUC     -> quantifies what the augmented replicas bought "
      "against a REAL cross-dataset benchmark (ablation rows above are FakeAVCeleb-test only).")

### Cell 9 (NEW) — Paper Artifact Export
Collects every JSON/CSV this notebook produced into `results_v33.json` and generates ready-to-paste
LaTeX for the validation, DFDC-generalization, and ablation tables. All figures were already saved at
300 dpi in Cells 5/6/8.

In [ ]:
# =====================================================================
# CELL 9 (NEW in v3.3): PAPER ARTIFACT EXPORT
# Collects every JSON/CSV this notebook wrote into one results_v33.json,
# and generates ready-to-paste LaTeX for the paper's headline tables.
# All figures were already saved at 300 dpi in Cells 5/6/8.
# =====================================================================

import json
import os
import csv

def _load_json(path, default=None):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    print(f"[!] Missing: {path} (run the corresponding cell first)")
    return default

def _load_csv(path):
    if os.path.exists(path):
        with open(path, newline="") as f:
            return list(csv.DictReader(f))
    print(f"[!] Missing: {path} (run Cell 8 first)")
    return []

best_metrics_j = _load_json("/kaggle/working/best_metrics.json", {})
calibration_j = _load_json("/kaggle/working/calibration.json", {})
dfdc_results_j = _load_json("/kaggle/working/dfdc_results_v33.json", {})
ablation_rows_j = _load_csv("/kaggle/working/ablation_results.csv")

results_v33 = {
    "paper_title": "CrossFuse: Multi-Modal Deepfake Detection Using Cross-Attention Fusion of "
                    "Audio and Visual Streams",
    "notebook_version": "v3.3",
    "config": CONFIG,
    "training": best_metrics_j,
    "calibration": calibration_j,
    "dfdc_zero_shot": dfdc_results_j,
    "ablations": ablation_rows_j,
}
with open("/kaggle/working/results_v33.json", "w") as f:
    json.dump(results_v33, f, indent=2)
print("results_v33.json written.")


def fmt(x, nd=4):
    try:
        return f"{float(x):.{nd}f}"
    except (TypeError, ValueError):
        return "--"

# ---- Table 2: Val (calibration-time) per-head performance ----
table2 = r"""
\begin{table}[t]
\centering
\caption{CrossFuse v3.3 validation performance (identity-disjoint, checkpoint selection).}
\begin{tabular}{lccc}
\toprule
Head & Val AUC & Best Epoch & Note \\
\midrule
Video & %s & %s & masked stats pooling \\
Audio & %s & %s & CMVN-normalized MFCC \\
AV-Sync & %s & %s & self-supervised, shuffled pairs \\
\bottomrule
\end{tabular}
\end{table}
""" % (fmt(best_metrics_j.get("best_val_video_auc")), best_metrics_j.get("best_epoch", "--"),
       fmt(best_metrics_j.get("best_val_audio_auc")), best_metrics_j.get("best_epoch", "--"),
       fmt(best_metrics_j.get("best_val_sync_auc")), best_metrics_j.get("best_epoch", "--"))

# ---- Table 3: DFDC cross-dataset generalization ----
gap = dfdc_results_j.get("generalization_gap")
ci = dfdc_results_j.get("video_auc_ci", [None, None])
table3 = r"""
\begin{table}[t]
\centering
\caption{Zero-shot cross-dataset generalization on DFDC (video head, %s real / %s fake, bootstrap 95\%% CI).}
\begin{tabular}{lc}
\toprule
Metric & Value \\
\midrule
DFDC AUC (transferred threshold) & %s [%s, %s] \\
DFDC Accuracy (transferred threshold) & %s \\
DFDC ECE (post-temperature) & %s \\
Val $\to$ DFDC Generalization Gap & %s \\
\bottomrule
\end{tabular}
\end{table}
""" % (dfdc_results_j.get("n_real", "--"), dfdc_results_j.get("n_fake", "--"),
       fmt(dfdc_results_j.get("video_auc_mean")), fmt(ci[0]), fmt(ci[1]),
       fmt(dfdc_results_j.get("video_acc_transferred_thr")), fmt(dfdc_results_j.get("video_ece")), fmt(gap))

# ---- Table 4: Ablation grid ----
ablation_lines = "\n".join(
    r"%s & %s & %s & %s \\" % (
        r["row"].replace("_", r"\_"), fmt(r.get("video_auc")), fmt(r.get("audio_auc")), fmt(r.get("combined_acc")))
    for r in ablation_rows_j
)
table4 = r"""
\begin{table}[t]
\centering
\caption{Ablation study (FakeAVCeleb held-out test split).}
\begin{tabular}{lccc}
\toprule
Configuration & Video AUC & Audio AUC & Combined Acc. \\
\midrule
%s
\bottomrule
\end{tabular}
\end{table}
""" % ablation_lines

for name, tex in [("table2_val_perhead", table2), ("table3_dfdc_generalization", table3),
                   ("table4_ablations", table4)]:
    path = f"/kaggle/working/{name}.tex"
    with open(path, "w") as f:
        f.write(tex)
    print(f"Wrote {path}")

print("\n" + "=" * 65)
print("Table 2 (val per-head) preview:")
print(table2)
print("Table 3 (DFDC generalization) preview:")
print(table3)
print("Table 4 (ablations) preview:")
print(table4)

print("\nAll figures were already saved at 300 dpi:")
for fig in ["crossfuse_v33_perhead_plots.png", "crossfuse_v33_sync_roc.png",
            "crossfuse_v33_attribution_matrix.png", "dfdc_generalization_matrix_v33.png"]:
    p = f"/kaggle/working/{fig}"
    print(f"  {'[OK]' if os.path.exists(p) else '[MISSING]'} {p}")

print("\nv3.3 paper artifact export complete.")